# Task 2: Yelp Polarity Sentiment Classification (Parth)

DATA 266 Lab 1, Task 2.

## 1. Objective

I classify Yelp reviews as negative (0) or positive (1) and compare a baseline with two neural models whose embeddings are learned from scratch:

| Name in code | Model | Role |
|---|---|---|
| `logreg` | TF-IDF (word 1-2 grams) + logistic regression | Baseline (model A) |
| `neural` | Embeddings learned from scratch + mean pooling + MLP | Neural model B |
| `neural_cnn` | Embeddings learned from scratch + TextCNN (kernels 3, 4, 5) | Neural model C |

The notebook covers data loading and cleaning (section 3), preprocessing and vocabulary (4 to 6), the three models (7 to 9), training (10, 11), evaluation and statistical comparison (12, 13), the results (15), the review of 20 misclassified examples (16) and the discussion (17, 18).

**Running it.** Select the project kernel and use *Restart & Run All*. The notebook prepares the data, trains the three models, evaluates them on the full test set and shows the results. It needs the packages in `requirements.txt` and either `train.csv` / `test.csv` in `task2_sentiment/data/` or network access to the Hugging Face dataset.

**Outputs.** Everything the notebook writes (processed data, checkpoints, predictions, figures, logs, manifests, metrics report) goes to `task2_sentiment/parth/notebook_runs/<RUN_TAG>/`. Nothing outside that folder is changed.

The next cell holds the only settings meant to be edited.

In [ ]:
import os

RUN_MODE = os.environ.get("TASK2_RUN_MODE", "full")  # "full": the reported experiment; "smoke": tiny synthetic data, end-to-end check
RUN_TAG = os.environ.get("TASK2_RUN_TAG", "gpu_run")  # outputs go to task2_sentiment/parth/notebook_runs/<RUN_TAG>/
VERIFY_REPORTED_CHECKPOINTS = os.environ.get("TASK2_VERIFY_REPORTED", "1") == "1"  # also re-evaluate the reported checkpoints (section 15)
ALLOW_CPU_FULL_RUN = os.environ.get("TASK2_ALLOW_CPU", "0") == "1"  # the neural models are meant for a CUDA GPU

assert RUN_MODE in ("full", "smoke"), RUN_MODE

## 2. Imports and reproducibility

`CUBLAS_WORKSPACE_CONFIG` must be set before PyTorch initialises CUDA, so it is the first statement of the imports cell. Figures are rendered with the non-interactive Agg backend and displayed as PNG images. The repository root is found by walking up from the working directory, so the notebook runs the same from the repository root or from `src/`; every path written to a log, manifest, or table is relative to it.

In [ ]:
# Why: cuBLAS needs this set before CUDA initialises for deterministic matmuls.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import copy
import hashlib
import io
import json
import logging
import math
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
from collections import Counter
from datetime import datetime
from functools import lru_cache
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # Why: lab machines and CI have no display.

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
from IPython.display import Image, Markdown, display
from scipy.stats import binomtest, chi2
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix, f1_score, log_loss,
                             matthews_corrcoef, precision_recall_curve, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from torch.utils.data import DataLoader, Dataset

# Saved outputs must not show machine-specific paths, which the default warning format prints.
warnings.formatwarning = lambda message, category, *args, **kwargs: f"{category.__name__}: {message}\n"
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

### Paths and configuration helpers

In [ ]:
def find_repo_root(marker: str = "task2_sentiment/parth") -> Path:
    """The repository root is the closest folder above the working directory that contains task2_sentiment/parth."""
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / marker).is_dir():
            return base
    raise FileNotFoundError(f"{marker}/ not found above the working directory; start Jupyter inside the repository.")


REPO_ROOT = find_repo_root()

LABEL_NAMES = {0: "negative", 1: "positive"}


def resolve_path(p):
    """Return an absolute path; relative paths are anchored at the repo root."""
    p = Path(p)
    return p if p.is_absolute() else REPO_ROOT / p


def load_config(path):
    """Load the YAML config from a path given relative to cwd or repo root."""
    p = Path(path)
    if not p.exists():
        p = resolve_path(path)
    with open(p, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def apply_smoke_overrides(cfg):
    """Return a copy of cfg shrunk for smoke runs (tiny data, 1-2 epochs, no workers)."""
    cfg = copy.deepcopy(cfg)
    s = cfg.get("smoke", {})
    cfg["evaluation"]["bootstrap_samples"] = s.get("bootstrap_samples", 50)
    cfg["evaluation"]["min_slice_n"] = 5
    cfg["tokenizer"]["min_freq"] = 1
    for m in cfg["models"].values():
        if m["type"] == "neural":
            m["epochs"] = s.get("epochs", 2)
            m["batch_size"] = s.get("batch_size", 32)
            m["num_workers"] = 0
        elif m["type"] == "logreg":
            m["tfidf"]["min_df"] = 1
            m["max_iter"] = 200
    return cfg


def get_paths(cfg, smoke=False):
    """Build (and create) the output directories for a real or smoke run."""
    # Why: smoke runs write to a separate folder so they can never overwrite
    # real checkpoints, metrics, or logs.
    root = resolve_path(cfg["paths"]["smoke_dir"] if smoke else cfg["paths"]["member_dir"])
    paths = {
        "root": root,
        "data_dir": resolve_path(cfg["paths"]["data_dir"]),
        "data_processed": root / "data_processed",
        "checkpoints": root / "checkpoints",
        "outputs": root / "outputs",
        "figures": root / "figures",
        "logs": root / "logs",
        "metrics_report": root / "metrics_report.csv",
    }
    for k in ["data_processed", "checkpoints", "outputs", "figures", "logs"]:
        paths[k].mkdir(parents=True, exist_ok=True)
    return paths


def get_model_cfg(cfg, model_name):
    if model_name not in cfg["models"]:
        raise KeyError(f"Model '{model_name}' not in config. Available: {list(cfg['models'])}")
    return cfg["models"][model_name]


def checkpoint_paths(paths, model_name, model_type):
    """File names follow the lab spec: logreg_tfidf.joblib, best_neural.pt, latest_neural.pt."""
    if model_type == "logreg":
        return {"model": paths["checkpoints"] / f"{model_name}_tfidf.joblib"}
    return {
        "best": paths["checkpoints"] / f"best_{model_name}.pt",
        "latest": paths["checkpoints"] / f"latest_{model_name}.pt",
    }

### Seeding, device, hardware, memory, and timing

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Why: warn_only keeps training running if an op has no deterministic kernel,
    # but the warning still lands in the raw log.
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.benchmark = False


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def hardware_info():
    info = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu": platform.processor() or platform.machine(),
        "cpu_count": os.cpu_count(),
        "torch": torch.__version__,
        "torch_cuda_build": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": None,
        "gpu_count": 0,
        "gpu_capability": None,
        "gpu_total_mem_gb": None,
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info.update(
            gpu_name=torch.cuda.get_device_name(0),
            gpu_count=torch.cuda.device_count(),
            gpu_capability=f"sm_{props.major}{props.minor}",
            gpu_total_mem_gb=round(props.total_memory / 1024**3, 2),
        )
    return info


def device_label(info, device):
    """Human-readable name of the device a stage actually ran on."""
    if device.type == "cuda" and info["gpu_name"]:
        return f"{info['gpu_name']} ({info['gpu_capability']}, {info['gpu_total_mem_gb']} GB)"
    return f"CPU ({info['cpu']}, {info['cpu_count']} cores)"


def reset_peak_memory(device):
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)


def peak_memory_mb(device):
    """GPU peak allocated by PyTorch in this phase, plus process-lifetime peak RSS."""
    gpu = None
    if device.type == "cuda":
        gpu = round(torch.cuda.max_memory_allocated(device) / 1024**2, 1)
    cpu = None
    try:
        import resource  # Linux/macOS only

        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        # Why: Linux reports KB, macOS reports bytes.
        cpu = round(rss / 1024**2, 1) if sys.platform == "darwin" else round(rss / 1024, 1)
    except ImportError:
        pass
    return {"gpu_peak_mb": gpu, "cpu_peak_rss_mb": cpu}


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self.start


def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

### Logging, file helpers, and run manifests

In [ ]:
def setup_logger(name, log_path):
    """Log to console and to a new timestamped file that is never rewritten."""
    logger = logging.getLogger(f"{name}_{log_path}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(sh)
    return logger


def to_builtin(obj):
    """Convert numpy scalars/arrays into plain Python types for JSON and torch.save."""
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, Path):
        return str(obj)
    return obj


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_builtin(obj), f, indent=2)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def rel(path):
    """Repo-relative string for logs and CSVs, so no personal paths get committed."""
    try:
        return str(Path(path).resolve().relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def config_hash(obj):
    return hashlib.sha256(json.dumps(to_builtin(obj), sort_keys=True).encode()).hexdigest()[:12]


def upsert_csv_row(path, row, key_cols):
    """Insert or replace one row (matched on key_cols) so reruns don't duplicate models."""
    new = pd.DataFrame([to_builtin(row)])
    if Path(path).exists():
        old = pd.read_csv(path)
        if all(k in old.columns for k in key_cols):
            mask = np.ones(len(old), dtype=bool)
            for k in key_cols:
                mask &= old[k].astype(str) == str(row[k])
            old = old[~mask]
        new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def package_versions():
    names = ["torch", "numpy", "pandas", "scikit-learn", "scipy", "nltk", "PyYAML",
             "matplotlib", "joblib", "datasets"]
    out = {}
    try:
        from importlib.metadata import PackageNotFoundError, version
    except ImportError:  # pragma: no cover
        return out
    for n in names:
        try:
            out[n] = version(n)
        except PackageNotFoundError:
            out[n] = None
    return out


def git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return None


def write_manifest(paths, model_name, stage, extra):
    """Record versions, config, hardware, and checkpoint hashes for one run stage."""
    manifest = {
        "model": model_name,
        "stage": stage,
        "created": datetime.now().isoformat(timespec="seconds"),
        "git_commit": git_commit(),
        "packages": package_versions(),
        "hardware": hardware_info(),
        **extra,
    }
    # Why: timestamped like the raw logs, so an earlier run's manifest is never overwritten.
    out = paths["logs"] / f"manifest_{stage}_{model_name}_{timestamp()}.json"
    save_json(manifest, out)
    return out

In [ ]:
def show_table(rows, **kwargs):
    display(pd.DataFrame(rows, **kwargs))


def show_fig(fig):
    """Render a matplotlib figure as a PNG (the Agg backend does not display figures by itself) and close it."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buf.getvalue()))


def cpu_model_name():
    try:
        if sys.platform.startswith("linux"):
            for line in Path("/proc/cpuinfo").read_text().splitlines():
                if line.lower().startswith("model name"):
                    return line.split(":", 1)[1].strip()
        elif sys.platform == "darwin":
            out = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True)
            if out.stdout.strip():
                return out.stdout.strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()

### Seed and hardware

The run seed is 8503. `set_seed` seeds Python, NumPy and PyTorch, turns on deterministic algorithms and disables the cuDNN autotuner; the training and evaluation functions call it again before they start. The logistic regression runs on the CPU and the neural models on the GPU; the device each stage used is recorded in the training summaries and in `metrics_report.csv`.

The exact CPU model of the machine that trained the reported logistic regression was not recorded: the logs only record `x86_64, 24 cores`, because Python's `platform.processor()` does not return the model name on Linux.

In [ ]:
SEED = 8503
set_seed(SEED)
DEVICE = get_device()
HW = hardware_info()
show_table([(k, str(v)) for k, v in HW.items()] + [("CPU model", cpu_model_name()), ("run mode", RUN_MODE)],
           columns=["item", "value"])

if RUN_MODE == "full" and DEVICE.type != "cuda" and not ALLOW_CPU_FULL_RUN:
    raise RuntimeError("RUN_MODE='full' trains the reported neural models and needs a CUDA GPU. Run on the GPU "
                       "machine, set RUN_MODE='smoke' for a small CPU check, or set TASK2_ALLOW_CPU=1 to force CPU.")

### Configuration

All settings of the experiment are in `CONFIG` below. The values are those of the reported runs. The cell after it asserts that they are identical to `config/train.yaml` and to the config recorded in each reported training manifest, whenever those files are present. Only `paths.member_dir` and `paths.smoke_dir` differ: they point into this notebook's run folder.

In [ ]:
MEMBER_DIR = REPO_ROOT / "task2_sentiment" / "parth"
RUN_DIR = MEMBER_DIR / "notebook_runs" / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_REL = rel(RUN_DIR)

CONFIG = {
    "seed": SEED,
    "paths": {
        "data_dir": "task2_sentiment/data",
        "member_dir": RUN_REL,
        "smoke_dir": f"{RUN_REL}/smoke_runs",
    },
    "data": {
        "hf_names": ["fancyzhx/yelp_polarity", "yelp_polarity"],
        "val_fraction": 0.05,
        "train_subset": None,
        "test_subset": None,
    },
    "preprocessing": {
        "lowercase": True,
        "remove_punctuation": True,
        "remove_stopwords": True,
        "keep_stopwords": ["not", "no", "nor", "never", "nothing", "none", "cannot", "neither", "nobody", "noone",
                           "nowhere"],
        "stemming": True,
        "min_tokens": 1,
    },
    "tokenizer": {"max_vocab": 50000, "min_freq": 2, "max_length": 256},
    "eda": {"length_bins": 60},
    "slices": {
        "short_max_words": 50,
        "long_min_words": 300,
        "negation_words": ["not", "no", "never", "nothing", "none", "nobody", "neither", "nor", "cannot", "without"],
        "contrast_words": ["but", "however", "although", "though", "except", "yet", "unfortunately"],
    },
    "evaluation": {
        "threshold": 0.5,
        "ece_bins": 15,
        "bootstrap_samples": 1000,
        "baseline_model": "logreg",
        "error_review_k": 5,
        "min_slice_n": 100,
    },
    "models": {
        "logreg": {
            "type": "logreg",
            "tfidf": {"ngram_range": [1, 2], "max_features": 200000, "min_df": 3, "sublinear_tf": True},
            "C": 1.0,
            "max_iter": 1000,
            "solver": "lbfgs",
        },
        "neural": {
            "type": "neural",
            "arch": "meanpool_mlp",
            "embedding_dim": 128,
            "hidden_dim": 256,
            "dropout": 0.2,
            "batch_size": 128,
            "eval_batch_size": 512,
            "learning_rate": 0.0003,
            "weight_decay": 0.01,
            "epochs": 5,
            "grad_clip": 1.0,
            "num_workers": 2,
            "select_by": "val_loss",
            "log_every": 200,
        },
        "neural_cnn": {
            "type": "neural",
            "arch": "textcnn",
            "embedding_dim": 128,
            "num_filters": 128,
            "kernel_sizes": [3, 4, 5],
            "dropout": 0.3,
            "batch_size": 128,
            "eval_batch_size": 512,
            "learning_rate": 0.0003,
            "weight_decay": 0.01,
            "epochs": 5,
            "grad_clip": 1.0,
            "num_workers": 2,
            "select_by": "val_loss",
            "log_every": 200,
        },
    },
    "smoke": {"n_train": 240, "n_test": 80, "epochs": 2, "batch_size": 32, "bootstrap_samples": 50},
}

In [ ]:
def without_run_paths(c):
    c = copy.deepcopy(c)
    paths = c.pop("paths")
    return c, paths["data_dir"]


def train_manifests(model_name, logs_dir):
    pattern = re.compile(rf"manifest_train_{re.escape(model_name)}_\d{{8}}_\d{{6}}\.json$")
    return sorted(p for p in logs_dir.glob(f"manifest_train_{model_name}_*.json") if pattern.match(p.name))


consistency = []
yaml_path = MEMBER_DIR / "config" / "train.yaml"
if yaml_path.exists():
    consistency.append(("config/train.yaml", without_run_paths(load_config(yaml_path)) == without_run_paths(CONFIG)))
for m in CONFIG["models"]:
    found = train_manifests(m, MEMBER_DIR / "logs")
    if found:
        recorded = load_json(found[-1])["config"]
        consistency.append((f"training manifest of {m}: {found[-1].name}",
                            without_run_paths(recorded) == without_run_paths(CONFIG)))
show_table(consistency, columns=["reference", "identical (output folders excluded)"])
assert all(ok for _, ok in consistency), "the notebook config differs from the reported configuration"

MODELS = list(CONFIG["models"])
BASELINE = CONFIG["evaluation"]["baseline_model"]
SMOKE = RUN_MODE == "smoke"
RUN_CFG = apply_smoke_overrides(CONFIG) if SMOKE else CONFIG
PATHS = get_paths(RUN_CFG, smoke=SMOKE)
print("models:", MODELS, "| baseline:", BASELINE, "| outputs:", rel(PATHS["root"]))

## 3. Dataset loading and cleaning

Yelp Polarity: the official train split (560,000 reviews) and test split (38,000 reviews). `load_raw_splits` uses `train.csv` / `test.csv` in `task2_sentiment/data/` if present, otherwise the Hugging Face dataset named in the config. `normalize_labels` maps both sources to 0 = negative, 1 = positive. `clean_split` drops missing, invalid, and empty rows; in the training split only, exact duplicate texts are dropped and texts that appear with both labels are removed entirely. The official test set is left untouched. A stratified 5% of the cleaned training split becomes the validation set (section 6).

`make_synthetic_df` generates the small labelled corpus used only by `RUN_MODE = "smoke"`.

In [ ]:
def read_yelp_csv(path):
    """Read a Yelp CSV with or without a header; returns columns text, label."""
    df = pd.read_csv(path, header=None, dtype=str, keep_default_na=True)
    first = [str(v).strip().lower() for v in df.iloc[0].tolist()]
    if "text" in first and "label" in first:  # file has a header row
        df.columns = first
        df = df.iloc[1:]
    else:  # fast.ai format: label, text with no header
        df = df.iloc[:, :2]
        df.columns = ["label", "text"]
    return df[["text", "label"]].reset_index(drop=True)


def normalize_labels(series):
    """Map labels to 0=negative, 1=positive. Accepts {0,1} or {1,2}; invalid -> NaN."""
    y = pd.to_numeric(series, errors="coerce")
    valid = set(y.dropna().unique().tolist())
    if valid <= {0, 1}:
        return y
    if valid <= {1, 2}:
        return y - 1
    # Mixed or unexpected values: keep only 1/2 style if that is the majority
    if y.isin([1, 2]).mean() >= y.isin([0, 1]).mean():
        return (y - 1).where(y.isin([1, 2]))
    return y.where(y.isin([0, 1]))


def load_raw_splits(cfg, paths, logger):
    """Prefer local CSVs in task2_sentiment/data/, otherwise download from Hugging Face."""
    train_csv, test_csv = paths["data_dir"] / "train.csv", paths["data_dir"] / "test.csv"
    if train_csv.exists() and test_csv.exists():
        logger.info(f"Loading local CSVs from {rel(paths['data_dir'])}")
        return read_yelp_csv(train_csv), read_yelp_csv(test_csv), "local_csv"
    try:
        from datasets import load_dataset
    except ImportError as e:
        raise FileNotFoundError(
            f"No CSVs at {rel(train_csv)} / {rel(test_csv)} and `datasets` is not installed."
        ) from e
    last_err = None
    for name in cfg["data"]["hf_names"]:
        try:
            logger.info(f"Loading Hugging Face dataset '{name}'")
            ds = load_dataset(name)
            return ds["train"].to_pandas(), ds["test"].to_pandas(), f"hf:{name}"
        except Exception as e:  # try the next alias
            last_err = e
            logger.warning(f"Could not load '{name}': {e}")
    raise RuntimeError(f"Could not load Yelp Polarity from any source: {last_err}")

In [ ]:
def clean_split(df, split, drop_duplicates):
    """Drop missing/empty text and invalid labels; report counts for the EDA summary."""
    report = {"split": split, "rows_raw": int(len(df))}
    df = df.copy()
    df["label"] = normalize_labels(df["label"])
    report["missing_text"] = int(df["text"].isna().sum())
    report["invalid_label"] = int(df["label"].isna().sum())
    df = df.dropna(subset=["text", "label"])
    df["text"] = df["text"].map(normalize_raw)
    empty = df["text"].str.len() == 0
    report["empty_text"] = int(empty.sum())
    df = df[~empty]
    dup_mask = df.duplicated(subset=["text"], keep=False)
    conflicting = df[dup_mask].groupby("text")["label"].nunique()
    conflicting = set(conflicting[conflicting > 1].index)
    report["duplicate_texts"] = int(df.duplicated(subset=["text"]).sum())
    report["conflicting_label_texts"] = len(conflicting)
    if drop_duplicates:
        # Why: only the training split is de-duplicated. The official test set is
        # left untouched so results stay comparable with other members.
        df = df[~df["text"].isin(conflicting)]
        df = df.drop_duplicates(subset=["text"])
    df["label"] = df["label"].astype(int)
    report["rows_clean"] = int(len(df))
    return df.reset_index(drop=True), report

In [ ]:
_POS = ["amazing", "great", "delicious", "friendly", "excellent", "wonderful", "fresh", "perfect"]


_NEG = ["terrible", "awful", "rude", "bland", "horrible", "dirty", "slow", "disgusting"]


_NOUNS = ["food", "service", "staff", "pizza", "coffee", "place", "waiter", "price"]


_FILLER = ["we", "came", "here", "on", "a", "friday", "night", "with", "family", "and", "ordered"]


def make_synthetic_df(n, seed):
    """Tiny labelled review set for smoke runs and tests; roughly balanced, varied lengths."""
    rng = random.Random(seed)
    rows = []
    for i in range(n):
        label = i % 2
        words = _POS if label == 1 else _NEG
        parts = []
        for _ in range(rng.randint(1, 4)):
            parts.append(f"The {rng.choice(_NOUNS)} was {rng.choice(words)}.")
        if rng.random() < 0.3:  # some long reviews so every length slice is populated
            parts.append(" ".join(rng.choices(_FILLER, k=rng.randint(60, 320))) + ".")
        if rng.random() < 0.2:  # some negated opposite words
            opp = _NEG if label == 1 else _POS
            parts.append(f"It was not {rng.choice(opp)} at all, but the {rng.choice(_NOUNS)} was ok.")
        rows.append({"text": " ".join(parts), "label": label})
    return pd.DataFrame(rows)

## 4. Text preprocessing

`TextPreprocessor` applies these steps in this order, configured by `CONFIG["preprocessing"]`:

1. Undo CSV escape artifacts (literal `\n` and `\"` in the Yelp files).
2. Lowercase.
3. Remove punctuation and special characters (apostrophes are dropped so that "don't" becomes "dont").
4. Split on whitespace (tokenization).
5. Remove scikit-learn's English stopwords, except the negation words in `keep_stopwords`.
6. Porter stemming (NLTK).

In [ ]:
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"


PAD_ID, UNK_ID = 0, 1


_ESCAPED_NEWLINE = re.compile(r"\\n")      # Yelp CSVs store newlines as a literal "\n"


_ESCAPED_QUOTE = re.compile(r'\\"')


_APOSTROPHE = re.compile(r"[\u2019']")


_NON_ALNUM_LOWER = re.compile(r"[^a-z0-9\s]")


_NON_ALNUM_ANY = re.compile(r"[^A-Za-z0-9\s]")


def normalize_raw(text):
    """Undo CSV escape artifacts only; used for display text and slice detection."""
    text = _ESCAPED_NEWLINE.sub(" ", str(text))
    return _ESCAPED_QUOTE.sub('"', text).strip()

In [ ]:
class TextPreprocessor:
    """Turns a raw review into a list of cleaned tokens according to cfg['preprocessing']."""

    def __init__(self, pp_cfg):
        self.cfg = pp_cfg
        keep = set(str(w).lower() for w in pp_cfg.get("keep_stopwords", []))
        # Why: the stock English stopword list contains negations ("not", "no",
        # "never"). Deleting them flips "not good" into "good", which directly
        # hurts sentiment, so the config can protect them.
        self.stopwords = set(ENGLISH_STOP_WORDS) - keep
        self._stem = None
        if pp_cfg.get("stemming", False):
            from nltk.stem import PorterStemmer

            # Why: Yelp has far fewer unique words than tokens, so caching the
            # stemmer turns ~70M stem calls into a few hundred thousand.
            self._stem = lru_cache(maxsize=None)(PorterStemmer().stem)

    def __call__(self, text):
        t = normalize_raw(text)
        if self.cfg.get("lowercase", True):
            t = t.lower()
        if self.cfg.get("remove_punctuation", True):
            t = _APOSTROPHE.sub("", t)  # "don't" -> "dont" instead of "don t"
            t = (_NON_ALNUM_LOWER if self.cfg.get("lowercase", True) else _NON_ALNUM_ANY).sub(" ", t)
        tokens = t.split()
        if self.cfg.get("remove_stopwords", True):
            tokens = [w for w in tokens if w.lower() not in self.stopwords]
        if self._stem is not None:
            tokens = [self._stem(w) for w in tokens]
        return tokens

    def to_string(self, text):
        return " ".join(self(text))

In [ ]:
pp = CONFIG["preprocessing"]
example = "The food was NOT good!!\\nWe didn't love the waiters, but the dessert was amazing."
stages = [
    ("raw text", None),
    ("escapes undone + lowercase", {**pp, "remove_punctuation": False, "remove_stopwords": False, "stemming": False}),
    ("+ punctuation removed", {**pp, "remove_stopwords": False, "stemming": False}),
    ("+ stopwords removed (negations kept)", {**pp, "stemming": False}),
    ("+ Porter stemming (final tokens)", pp),
]
for name, stage_cfg in stages:
    text = example if stage_cfg is None else TextPreprocessor(stage_cfg).to_string(example)
    print(f"{name:<40} {text}")

protected = [w for w in pp["keep_stopwords"] if w in ENGLISH_STOP_WORDS]
print("\nnegation words protected from scikit-learn's stopword list:", protected)
print("'not good' ->", TextPreprocessor(pp)("not good"),
      "| without the protection ->", TextPreprocessor({**pp, "keep_stopwords": []})("not good"))

## 5. Vocabulary and word tokenizer

`WordTokenizer.build` counts words in the training split only, keeps those seen at least `min_freq` times, and caps the vocabulary at `max_vocab` entries including `<pad>` (id 0) and `<unk>` (id 1). `encode` keeps the first `max_length` tokens of a review and maps unseen words to `<unk>`. The TF-IDF baseline uses the same preprocessed token strings with its own vectorizer.

In [ ]:
class WordTokenizer:
    """Word-level vocabulary: id 0 is padding, id 1 is unknown, then words by frequency."""

    def __init__(self, itos=None):
        self.itos = list(itos) if itos else [PAD_TOKEN, UNK_TOKEN]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    @classmethod
    def build(cls, token_strings, max_vocab, min_freq):
        """Count words in the training split only (never val/test) and keep the top ones."""
        counts = Counter()
        for s in token_strings:
            counts.update(s.split())
        words = [w for w, c in counts.most_common() if c >= min_freq]
        if max_vocab:
            words = words[: max_vocab - 2]
        return cls([PAD_TOKEN, UNK_TOKEN] + words)

    def __len__(self):
        return len(self.itos)

    def encode(self, token_string, max_length):
        ids = [self.stoi.get(w, UNK_ID) for w in token_string.split()[:max_length]]
        # Why: an empty review would give a zero-length mean pool; one <unk> keeps
        # the shape valid and acts as a neutral "no information" input.
        return ids if ids else [UNK_ID]

    def decode(self, ids):
        return [self.itos[i] for i in ids if i != PAD_ID]

In [ ]:
toy = WordTokenizer.build(["great food great staff", "bad food", "not great"], max_vocab=10, min_freq=1)
print("itos:", toy.itos)
print("encode('great pasta not bad', 3) ->", toy.encode("great pasta not bad", 3), "| decode ->", toy.decode(toy.encode("great pasta not bad", 3)))
print("encode('', 3) ->", toy.encode("", 3), "(an empty review becomes one <unk>)")

## 6. Data preparation, slices, and exploratory data analysis

`prepare_data` runs the whole pipeline: load, clean, carve the stratified validation split from the training split (seed 8503), preprocess every review, add the slice columns, drop training reviews that are empty after preprocessing, run the EDA, and cache the result in `data_processed/` (keyed by a hash of the data, preprocessing, and slice settings). Training and evaluation call it again and get the cached splits.

Slices used in the evaluation: raw length (short <= 50 words, long >= 300, otherwise medium), presence of a negation word (including contractions such as "didn't"), and presence of a contrast word.

In [ ]:
def _word_regex(words):
    # Why: str() guards against YAML turning an unquoted `no` into False.
    return re.compile(r"\b(?:" + "|".join(re.escape(str(w)) for w in words) + r")\b")


def add_slice_columns(df, slice_cfg):
    """Robustness slices computed on the raw text (before stopword removal)."""
    lower = df["text"].str.lower()
    neg_re = _word_regex(slice_cfg["negation_words"])
    con_re = _word_regex(slice_cfg["contrast_words"])
    df["n_words_raw"] = df["text"].str.split().str.len()
    df["length_bucket"] = np.where(
        df["n_words_raw"] <= slice_cfg["short_max_words"], "short",
        np.where(df["n_words_raw"] >= slice_cfg["long_min_words"], "long", "medium"))
    contraction_negation = lower.str.contains("n\'t", regex=False, na=False) | lower.str.contains("n’t", regex=False, na=False)
    df["has_negation"] = lower.str.contains(neg_re, regex=True, na=False) | contraction_negation
    df["has_contrast"] = lower.str.contains(con_re)
    return df


def slice_masks(df):
    """Named boolean masks used by evaluation for per-slice metrics."""
    return {
        "all": np.ones(len(df), dtype=bool),
        "length_short": (df["length_bucket"] == "short").to_numpy(),
        "length_medium": (df["length_bucket"] == "medium").to_numpy(),
        "length_long": (df["length_bucket"] == "long").to_numpy(),
        "has_negation": df["has_negation"].to_numpy(bool),
        "no_negation": ~df["has_negation"].to_numpy(bool),
        "has_contrast": df["has_contrast"].to_numpy(bool),
        "no_contrast": ~df["has_contrast"].to_numpy(bool),
    }

In [ ]:
def run_eda(splits, reports, cfg, paths, source):
    """Length distribution, class balance, missing/malformed counts, train-test overlap."""
    max_len = cfg["tokenizer"]["max_length"]
    summary = {"source": source, "cleaning": reports, "splits": {}}
    for name, df in splits.items():
        counts = df["label"].value_counts().sort_index()
        summary["splits"][name] = {
            "rows": int(len(df)),
            "class_counts": {int(k): int(v) for k, v in counts.items()},
            "positive_fraction": float(df["label"].mean()),
            "raw_words": df["n_words_raw"].describe(percentiles=[0.5, 0.9, 0.99]).to_dict(),
            "processed_tokens": df["n_tokens"].describe(percentiles=[0.5, 0.9, 0.99]).to_dict(),
            "fraction_truncated_at_max_length": float((df["n_tokens"] > max_len).mean()),
            "empty_after_preprocessing": int((df["n_tokens"] == 0).sum()),
            "length_bucket_counts": df["length_bucket"].value_counts().to_dict(),
            "has_negation_fraction": float(df["has_negation"].mean()),
            "has_contrast_fraction": float(df["has_contrast"].mean()),
        }
    overlap = set(splits["train"]["text"]) & set(splits["test"]["text"])
    summary["train_test_exact_overlap"] = len(overlap)
    save_json(summary, paths["outputs"] / "eda_summary.json")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, df in splits.items():
        axes[0].hist(df["n_words_raw"].clip(upper=1000), bins=cfg["eda"]["length_bins"],
                     alpha=0.5, label=name, density=True)
        axes[1].hist(df["n_tokens"].clip(upper=600), bins=cfg["eda"]["length_bins"],
                     alpha=0.5, label=name, density=True)
    axes[0].set_title("Raw review length (words, clipped at 1000)")
    axes[1].set_title("Processed length (tokens, clipped at 600)")
    axes[1].axvline(max_len, color="k", ls="--", label=f"max_length={max_len}")
    for ax in axes:
        ax.set_xlabel("length")
        ax.set_ylabel("density")
        ax.legend()
    fig.tight_layout()
    fig.savefig(paths["figures"] / "eda_review_length_distribution.png", dpi=120)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    names = list(splits)
    x = np.arange(len(names))
    for lbl, off in [(0, -0.2), (1, 0.2)]:
        vals = [int((splits[n]["label"] == lbl).sum()) for n in names]
        ax.bar(x + off, vals, width=0.4, label="negative (0)" if lbl == 0 else "positive (1)")
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylabel("reviews")
    ax.set_title("Class balance per split")
    ax.legend()
    fig.tight_layout()
    fig.savefig(paths["figures"] / "eda_class_balance.png", dpi=120)
    plt.close(fig)
    return summary

In [ ]:
def _process(df, pre, slice_cfg):
    df = df.copy()
    df["tokens"] = df["text"].map(pre.to_string)
    df["n_tokens"] = df["tokens"].str.split().str.len()
    return add_slice_columns(df, slice_cfg)


def prepare_data(cfg, paths, logger, smoke=False):
    """Return {'train','val','test'} DataFrames with columns id, text, label, tokens, slices."""
    seed = cfg["seed"]
    pre = TextPreprocessor(cfg["preprocessing"])
    if smoke:
        s = cfg["smoke"]
        train_raw = make_synthetic_df(s["n_train"], seed)
        test_raw = make_synthetic_df(s["n_test"], seed + 1)
        source = "synthetic"
    else:
        key = config_hash({k: cfg[k] for k in ["seed", "data", "preprocessing", "slices"]})
        meta_path = paths["data_processed"] / "meta.json"
        if meta_path.exists() and load_json(meta_path).get("key") == key:
            logger.info(f"Using cached processed data in {rel(paths['data_processed'])} (key={key})")
            return {s: pd.read_pickle(paths["data_processed"] / f"{s}.pkl") for s in ["train", "val", "test"]}
        train_raw, test_raw, source = load_raw_splits(cfg, paths, logger)

    train_df, r_train = clean_split(train_raw, "train", drop_duplicates=not smoke)
    test_df, r_test = clean_split(test_raw, "test", drop_duplicates=False)
    logger.info(f"Cleaning report: {r_train} | {r_test}")

    for name, df, n in [("train", train_df, cfg["data"].get("train_subset")),
                        ("test", test_df, cfg["data"].get("test_subset"))]:
        if n and not smoke and n < len(df):
            df = df.sample(n=n, random_state=seed).reset_index(drop=True)
            logger.info(f"Subsampled {name} to {n} rows")
        if name == "train":
            train_df = df
        else:
            test_df = df

    # Why: validation is carved from train (stratified) so the test set is only
    # touched once, at final evaluation.
    tr_idx, va_idx = train_test_split(np.arange(len(train_df)), test_size=cfg["data"]["val_fraction"],
                                      random_state=seed, stratify=train_df["label"])
    splits = {"train": train_df.iloc[tr_idx], "val": train_df.iloc[va_idx], "test": test_df}

    with Timer() as t:
        for name in splits:
            splits[name] = _process(splits[name], pre, cfg["slices"]).reset_index(drop=True)
            splits[name].insert(0, "id", [f"{name}_{i}" for i in range(len(splits[name]))])
    logger.info(f"Preprocessing took {t.seconds:.1f}s")

    # Reviews that become empty after preprocessing carry no signal for training.
    n_before = len(splits["train"])
    splits["train"] = splits["train"][splits["train"]["n_tokens"] >= cfg["preprocessing"]["min_tokens"]]
    splits["train"] = splits["train"].reset_index(drop=True)
    r_train["dropped_empty_after_preprocessing"] = n_before - len(splits["train"])

    run_eda(splits, [r_train, r_test], cfg, paths, source)
    if not smoke:
        for s, df in splits.items():
            df.to_pickle(paths["data_processed"] / f"{s}.pkl")
        save_json({"key": key, "source": source, "rows": {s: len(d) for s, d in splits.items()}},
                  paths["data_processed"] / "meta.json")
    logger.info("Split sizes: " + ", ".join(f"{s}={len(d)}" for s, d in splits.items()))
    return splits

In [ ]:
data_logger = setup_logger("data", PATHS["logs"] / f"data_{timestamp()}.log")
splits = prepare_data(RUN_CFG, PATHS, data_logger, smoke=SMOKE)
show_table([(s, len(df), round(float(df["label"].mean()), 4)) for s, df in splits.items()],
           columns=["split", "reviews", "positive fraction"])
display(splits["train"][["id", "label", "text", "tokens", "n_words_raw", "n_tokens", "length_bucket",
                         "has_negation", "has_contrast"]].head(3))

In [ ]:
eda = load_json(PATHS["outputs"] / "eda_summary.json")
print("source:", eda["source"], "| label names:", LABEL_NAMES)
display(pd.DataFrame(eda["cleaning"]).set_index("split"))
display(pd.DataFrame({
    s: {"rows": d["rows"], "negative (0)": d["class_counts"].get("0", d["class_counts"].get(0)),
        "positive (1)": d["class_counts"].get("1", d["class_counts"].get(1)),
        "positive fraction": round(d["positive_fraction"], 4)}
    for s, d in eda["splits"].items()
}).T)
print("exact text overlap between train and test:", eda["train_test_exact_overlap"])
display(Image(filename=str(PATHS["figures"] / "eda_class_balance.png")))

In [ ]:
max_len = CONFIG["tokenizer"]["max_length"]
length_rows = {}
for s, d in eda["splits"].items():
    rw, pt = d["raw_words"], d["processed_tokens"]
    length_rows[s] = {
        "mean raw words": rw["mean"], "median raw words": rw["50%"], "p90 raw words": rw["90%"],
        "p99 raw words": rw["99%"], "max raw words": rw["max"],
        "median processed tokens": pt["50%"], "p99 processed tokens": pt["99%"],
        f"fraction longer than max_length ({max_len})": d["fraction_truncated_at_max_length"],
        "empty after preprocessing": d["empty_after_preprocessing"],
    }
display(pd.DataFrame(length_rows).T.round(4))
display(Image(filename=str(PATHS["figures"] / "eda_review_length_distribution.png")))

display(pd.DataFrame({
    s: {**{f"length {k}": v for k, v in d["length_bucket_counts"].items()},
        "has negation (fraction)": round(d["has_negation_fraction"], 4),
        "has contrast (fraction)": round(d["has_contrast_fraction"], 4)}
    for s, d in eda["splits"].items()
}).T)

In [ ]:
tok_cfg = RUN_CFG["tokenizer"]
word_tok = WordTokenizer.build(splits["train"]["tokens"], tok_cfg["max_vocab"], tok_cfg["min_freq"])
print(f"vocabulary built from the training split: {len(word_tok):,} entries "
      f"(max_vocab={tok_cfg['max_vocab']}, min_freq={tok_cfg['min_freq']})")
print("first 25 entries:", word_tok.itos[:25])
demo_tokens = TextPreprocessor(pp).to_string(example)
demo_ids = word_tok.encode(demo_tokens, tok_cfg["max_length"])
print("tokens :", demo_tokens)
print("ids    :", demo_ids)
print("decoded:", word_tok.decode(demo_ids))

## 7. Model A: TF-IDF + logistic regression (baseline)

A scikit-learn pipeline: `TfidfVectorizer` over word 1-grams and 2-grams of the preprocessed text, then `LogisticRegression`. `train_logreg` fits it once on the training split on the CPU, scores the validation split and saves it as `logreg_tfidf.joblib`.

**Justification.** TF-IDF with logistic regression is a strong, standard baseline for sentiment classification. Unigrams carry most sentiment cues ("delicious", "rude"), bigrams add short patterns that single words miss, including negation pairs that survive preprocessing such as "not worth" or "never return", and sublinear TF-IDF down-weights terms that are frequent everywhere while keeping discriminative ones. A linear model over 200,000 sparse features with `min_df` 3 is well matched to 532K training reviews and fits in 58 s on the CPU.

It is also the best model here on every predictive metric (accuracy 0.9463, ROC-AUC 0.9876, MCC 0.8927, Brier 0.0424), so it is a demanding reference rather than a weak one. The lab's requirement to learn textual embeddings from scratch is met by the two neural models; the baseline uses no embeddings, only TF-IDF features fitted on the training split.

In [ ]:
def build_logreg(model_cfg, seed):
    """TF-IDF over the already-preprocessed token string, then logistic regression."""
    t = model_cfg["tfidf"]
    vec = TfidfVectorizer(
        lowercase=False,               # text is already lowercased in preprocessing
        token_pattern=r"(?u)\b\w+\b",  # keep 1-character tokens the preprocessor produced
        ngram_range=tuple(t["ngram_range"]),
        max_features=t["max_features"],
        min_df=t["min_df"],
        sublinear_tf=t["sublinear_tf"],
        dtype=np.float64,
    )
    clf = LogisticRegression(C=model_cfg["C"], max_iter=model_cfg["max_iter"],
                             solver=model_cfg.get("solver", "lbfgs"), random_state=seed)
    return Pipeline([("tfidf", vec), ("clf", clf)])


def logreg_param_count(pipeline):
    clf = pipeline.named_steps["clf"]
    return int(clf.coef_.size + clf.intercept_.size)

In [ ]:
def train_logreg(cfg, model_name, model_cfg, splits, paths, logger):
    device = torch.device("cpu")
    pipe = build_logreg(model_cfg, cfg["seed"])
    X_tr, y_tr = splits["train"]["tokens"].tolist(), splits["train"]["label"].to_numpy()
    X_va, y_va = splits["val"]["tokens"].tolist(), splits["val"]["label"].to_numpy()

    with Timer() as t:
        pipe.fit(X_tr, y_tr)
    logger.info(f"Fit finished in {t.seconds:.1f}s")

    p_tr = pipe.predict_proba(X_tr)[:, 1]
    p_va = pipe.predict_proba(X_va)[:, 1]
    summary = {
        "model": model_name, "type": "logreg",
        "param_count": logreg_param_count(pipe),
        "vocab_size": len(pipe.named_steps["tfidf"].vocabulary_),
        "train_loss": float(log_loss(y_tr, p_tr, labels=[0, 1])),
        "val_loss": float(log_loss(y_va, p_va, labels=[0, 1])),
        "val_accuracy": float(accuracy_score(y_va, (p_va >= 0.5).astype(int))),
        "converged_iterations": int(np.max(pipe.named_steps["clf"].n_iter_)),
        "training_time_s": t.seconds,
        "train_examples_per_sec": len(X_tr) / t.seconds,
        "epochs": 1,
    }
    # Why: TfidfVectorizer keeps every pruned n-gram in `stop_words_`, which can
    # make the saved file hundreds of MB; it is not needed for prediction.
    if getattr(pipe.named_steps["tfidf"], "stop_words_", None) is not None:
        pipe.named_steps["tfidf"].stop_words_ = None
    ckpt = checkpoint_paths(paths, model_name, "logreg")["model"]
    # Why: store the preprocessing settings with the model so prediction always
    # cleans text the same way the model was trained on.
    joblib.dump({"pipeline": pipe, "preprocessing_cfg": cfg["preprocessing"], "model_cfg": model_cfg}, ckpt)
    summary["checkpoint"] = rel(ckpt)
    summary["checkpoint_sha256"] = file_sha256(ckpt)
    return summary, device

In [ ]:
print(build_logreg(RUN_CFG["models"]["logreg"], SEED))

## 8. Model B: embeddings + mean pooling + MLP (`neural`)

Token embeddings learned from scratch, a masked mean over the non-padding positions, then a two-layer MLP with dropout that outputs two logits.

**Justification.** The MeanPoolMLP is the simplest neural model that learns embeddings from scratch. Each of the 50,000 vocabulary entries gets a 128-dimensional vector learned during training (6.4M of its 6,433,538 parameters), a masked mean over the non-padding positions turns a review of any length into one fixed-size vector, and a two-layer MLP (128 to 256 to 2 units, ReLU, dropout 0.2) learns nonlinear combinations of the averaged features.

The mean ignores order: "good food, terrible service" and "terrible food, good service" get the same representation, so phrases, negation scope and contrast are visible only through which words occur. That is a plausible reason why it trails both other models and has the highest error rate on the `has_contrast` slice (0.0776), but no run isolated it. The sizes (embedding 128, hidden 256, dropout 0.2, vocabulary 50,000, max length 256, 5 epochs) are a moderate configuration that trains in 73 s on the RTX 5090; no other setting was run.

In [ ]:
class MeanPoolMLP(nn.Module):
    """Embedding -> masked mean over tokens -> Linear -> ReLU -> Dropout -> Linear (2 logits)."""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout, num_classes=2):
        super().__init__()
        # Why: embeddings are randomly initialised and learned from Yelp only;
        # no pretrained embeddings are used.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask):
        emb = self.embedding(input_ids)                      # (B, T, E)
        mask = attention_mask.unsqueeze(-1).to(emb.dtype)    # (B, T, 1)
        # Why: dividing by the real token count, not T, stops padding from
        # shrinking the representation of short reviews.
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        h = self.dropout(self.act(self.fc1(self.dropout(pooled))))
        return self.fc2(h)

## 9. Model C: embeddings + TextCNN (`neural_cnn`)

Token embeddings learned from scratch, parallel 1-D convolutions with kernel sizes 3, 4 and 5 over the token sequence, ReLU, max-pooling over time (windows that contain only padding are masked out), dropout, and a linear layer to two logits. `build_neural_model` picks the class from the `arch` key of the model config.

**Justification.** The TextCNN keeps the same from-scratch embeddings but replaces the mean with parallel 1-D convolutions of width 3, 4 and 5 tokens, 128 filters each. Each filter acts as a learned detector for a short phrase pattern, the three widths cover phrases of different lengths, and max-over-time pooling keeps each filter's strongest match wherever it occurs in the review (windows that contain only padding are masked out). The 384 pooled features pass through dropout 0.3 (0.2 in the MeanPoolMLP) and a linear layer.

It beats the MeanPoolMLP on every metric in the table (accuracy 0.9379 against 0.9313, MCC 0.8759 against 0.8626), which fits the idea that local word order helps, although the two models also differ in dropout. It still trails the logistic regression, and each filter sees at most 5 processed tokens, so it cannot link a clause at the start of a review with a verdict at the end.

In [ ]:
class TextCNN(nn.Module):
    """Embedding -> parallel Conv1D branches (one per kernel size) -> ReLU ->
    masked global max pool over time -> concat -> Dropout -> Linear (2 logits).
    Same forward(input_ids, attention_mask) signature as MeanPoolMLP."""

    def __init__(self, vocab_size, embedding_dim, num_filters, kernel_sizes, dropout, num_classes=2):
        super().__init__()
        self.kernel_sizes = [int(k) for k in kernel_sizes]
        self.min_len = max(self.kernel_sizes)
        # Why: same from-scratch, randomly initialised embeddings as MeanPoolMLP.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        # Why: each branch detects a different n-gram width (3, 4, 5 tokens), so
        # word order inside short phrases matters, unlike mean pooling.
        self.convs = nn.ModuleList([nn.Conv1d(embedding_dim, num_filters, k) for k in self.kernel_sizes])
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(self.kernel_sizes), num_classes)

    def forward(self, input_ids, attention_mask):
        # Why: a width-5 kernel needs at least 5 positions, so batches of very
        # short reviews are right-padded up to the widest kernel.
        if input_ids.size(1) < self.min_len:
            extra = self.min_len - input_ids.size(1)
            input_ids = nn.functional.pad(input_ids, (0, extra), value=PAD_ID)
            attention_mask = nn.functional.pad(attention_mask, (0, extra), value=False)
        lengths = attention_mask.sum(1)                          # (B,) real tokens, padding is on the right
        x = self.embedding(input_ids).transpose(1, 2)            # (B, E, T)
        pooled = []
        for k, conv in zip(self.kernel_sizes, self.convs):
            h = self.act(conv(x))                                # (B, F, T-k+1)
            # Why: windows made only of padding still output ReLU(bias) and could
            # win the max, which would make a review's prediction depend on the
            # other reviews in its batch. Keep starts 0..L-k (start 0 if L < k).
            last = (lengths - k).clamp(min=0)
            pos = torch.arange(h.size(2), device=h.device)
            valid = pos.unsqueeze(0) <= last.unsqueeze(1)        # (B, T-k+1)
            h = h.masked_fill(~valid.unsqueeze(1), float("-inf"))
            pooled.append(h.max(dim=2).values)                   # (B, F)
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))

In [ ]:
# Register additional experimental architectures here: "arch_name": ClassName.
# Each class must accept (vocab_size, **arch kwargs from YAML) and implement
# forward(input_ids, attention_mask) -> logits of shape (B, 2).
NEURAL_MODELS = {"meanpool_mlp": MeanPoolMLP, "textcnn": TextCNN}


_NON_ARCH_KEYS = {"type", "arch", "batch_size", "eval_batch_size", "learning_rate", "weight_decay",
                  "epochs", "grad_clip", "num_workers", "select_by", "log_every", "description"}


def build_neural_model(model_cfg, vocab_size):
    arch = model_cfg.get("arch", "meanpool_mlp")
    if arch not in NEURAL_MODELS:
        raise KeyError(f"Unknown arch '{arch}'. Registered: {list(NEURAL_MODELS)}")
    kwargs = {k: v for k, v in model_cfg.items() if k not in _NON_ARCH_KEYS}
    return NEURAL_MODELS[arch](vocab_size=vocab_size, **kwargs)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
for name in [m for m in MODELS if RUN_CFG["models"][m]["type"] == "neural"]:
    net = build_neural_model(RUN_CFG["models"][name], len(word_tok))
    total, _ = count_parameters(net)
    display(Markdown(f"**{name}** with the {len(word_tok):,}-word vocabulary: {total:,} parameters"))
    print(net)
    show_table([(n, str(tuple(p.shape)), p.numel()) for n, p in net.named_parameters()] + [("total", "", total)],
               columns=["parameter", "shape", "count"])

## 10. Training utilities

- `EncodedReviews`: all token ids of a split in one flat array with offsets (low memory).
- `collate_batch` / `make_loader`: pad each batch to its own longest review, build the padding mask, and shuffle with a seeded generator.
- `evaluate_loss`: mean cross-entropy and accuracy on a loader, in evaluation mode.
- `plot_loss_curves`: per-epoch training and validation loss.

In [ ]:
class EncodedReviews(Dataset):
    """Stores all token ids in one flat int32 array plus offsets to keep memory low."""

    def __init__(self, token_strings, labels, tokenizer, max_length):
        seqs = [tokenizer.encode(s, max_length) for s in token_strings]
        self.lengths = np.array([len(s) for s in seqs], dtype=np.int64)
        self.offsets = np.concatenate([[0], np.cumsum(self.lengths)])
        self.flat = np.fromiter((i for s in seqs for i in s), dtype=np.int32, count=int(self.offsets[-1]))
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        ids = self.flat[self.offsets[i]:self.offsets[i + 1]]
        return torch.from_numpy(ids.astype(np.int64)), int(self.labels[i])


def collate_batch(batch):
    """Pad each batch only to its own longest review, and build the padding mask."""
    ids, labels = zip(*batch)
    padded = torch.nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=PAD_ID)
    return padded, padded != PAD_ID, torch.tensor(labels, dtype=torch.long)


def make_loader(dataset, batch_size, shuffle, seed, num_workers=0, pin_memory=False):
    gen = torch.Generator()
    gen.manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_batch,
                      num_workers=num_workers, pin_memory=pin_memory, generator=gen)

In [ ]:
@torch.no_grad()
def evaluate_loss(model, loader, device, loss_fn):
    model.eval()
    total, n, correct = 0.0, 0, 0
    for ids, mask, y in loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        logits = model(ids, mask)
        total += loss_fn(logits, y).item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    return total / n, correct / n


def plot_loss_curves(history, out_path, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ep = [h["epoch"] for h in history["epochs"]]
    axes[0].plot(ep, [h["train_loss"] for h in history["epochs"]], "o-", label="train")
    axes[0].plot(ep, [h["val_loss"] for h in history["epochs"]], "o-", label="validation")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("cross-entropy loss")
    axes[0].set_title(f"{title}: epoch loss")
    axes[0].legend()
    if history["steps"]:
        s = history["steps"]
        axes[1].plot([x["step"] for x in s], [x["train_loss"] for x in s], lw=1)
    axes[1].set_xlabel("optimizer step")
    axes[1].set_ylabel("train loss (running mean)")
    axes[1].set_title(f"{title}: step loss")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)

## 11. Training procedure

`train_neural` builds the vocabulary from the training split, re-seeds before initialisation, trains with cross-entropy and AdamW, clips gradients, skips non-finite steps, evaluates on the validation split after every epoch, saves `latest_<model>.pt` every epoch, and saves `best_<model>.pt` at the lowest validation loss. `train_model` is the entry point for one model: it sets up the log, seeds, prepares (or loads) the data, trains, and writes `train_summary_<model>.json` and a manifest. Both are called in section 14.

In [ ]:
def train_neural(cfg, model_name, model_cfg, splits, paths, logger):
    device = get_device()
    seed = cfg["seed"]
    tok_cfg = cfg["tokenizer"]
    max_len = tok_cfg["max_length"]

    with Timer() as t_tok:
        # Why: the vocabulary sees the training split only, so val/test words
        # the model never trained on correctly become <unk>.
        tokenizer = WordTokenizer.build(splits["train"]["tokens"], tok_cfg["max_vocab"], tok_cfg["min_freq"])
        train_ds = EncodedReviews(splits["train"]["tokens"], splits["train"]["label"], tokenizer, max_len)
        val_ds = EncodedReviews(splits["val"]["tokens"], splits["val"]["label"], tokenizer, max_len)
    logger.info(f"Vocab size {len(tokenizer)}; encoding took {t_tok.seconds:.1f}s")

    pin = device.type == "cuda"
    nw = model_cfg.get("num_workers", 0)
    train_loader = make_loader(train_ds, model_cfg["batch_size"], True, seed, nw, pin)
    val_loader = make_loader(val_ds, model_cfg.get("eval_batch_size", 512), False, seed, nw, pin)

    set_seed(seed)  # Why: re-seed right before init so weights match across reruns.
    model = build_neural_model(model_cfg, len(tokenizer)).to(device)
    total_params, trainable = count_parameters(model)
    logger.info(f"Model:\n{model}")
    logger.info(f"Parameters: total={total_params:,} trainable={trainable:,}")

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=model_cfg["learning_rate"],
                                  weight_decay=model_cfg["weight_decay"])
    ckpts = checkpoint_paths(paths, model_name, "neural")
    select_by = model_cfg.get("select_by", "val_loss")
    best_score = math.inf if select_by == "val_loss" else -math.inf
    history = {"epochs": [], "steps": []}
    log_every = model_cfg.get("log_every", 200)
    step, nan_steps, seen = 0, 0, 0

    reset_peak_memory(device)
    sync(device)
    with Timer() as t_train:
        for epoch in range(1, model_cfg["epochs"] + 1):
            model.train()
            ep_loss, ep_n, grad_norms, run = 0.0, 0, [], []
            with Timer() as t_ep:
                for ids, mask, y in train_loader:
                    ids = ids.to(device, non_blocking=pin)
                    mask = mask.to(device, non_blocking=pin)
                    y = y.to(device, non_blocking=pin)
                    loss = loss_fn(model(ids, mask), y)
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), model_cfg["grad_clip"])
                    if not torch.isfinite(loss) or not torch.isfinite(gn):
                        # Why: skip the update instead of letting one bad batch poison
                        # the weights; the count is logged as a stability metric.
                        nan_steps += 1
                        logger.warning(f"Non-finite loss/grad at step {step}; update skipped")
                        continue
                    optimizer.step()
                    step += 1
                    bs = y.size(0)
                    seen += bs
                    ep_loss += loss.item() * bs
                    ep_n += bs
                    grad_norms.append(gn.item())
                    run.append(loss.item())
                    if step % log_every == 0:
                        history["steps"].append({"step": step, "train_loss": float(np.mean(run))})
                        logger.info(f"epoch {epoch} step {step} train_loss {np.mean(run):.4f} "
                                    f"grad_norm {gn.item():.3f}")
                        run = []
                sync(device)
            val_loss, val_acc = evaluate_loss(model, val_loader, device, loss_fn)
            rec = {
                "epoch": epoch, "train_loss": ep_loss / max(ep_n, 1), "val_loss": val_loss,
                "val_accuracy": val_acc, "grad_norm_mean": float(np.mean(grad_norms)),
                "grad_norm_max": float(np.max(grad_norms)), "epoch_time_s": t_ep.seconds,
                "train_examples_per_sec": ep_n / t_ep.seconds,
            }
            history["epochs"].append(rec)
            logger.info("EPOCH " + " ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                                            for k, v in rec.items()))

            state = to_builtin({
                "model_name": model_name, "model_cfg": model_cfg, "tokenizer_cfg": tok_cfg,
                "preprocessing_cfg": cfg["preprocessing"], "seed": seed, "epoch": epoch,
                "history": history, "vocab": tokenizer.itos,
            })
            state["model_state"] = model.state_dict()
            state["optimizer_state"] = optimizer.state_dict()
            torch.save(state, ckpts["latest"])
            score = val_loss if select_by == "val_loss" else val_acc
            better = score < best_score if select_by == "val_loss" else score > best_score
            if better:
                best_score = score
                # Why: best_* is for evaluation only, so the optimizer state (2x the
                # model size with AdamW) stays in latest_* for resuming.
                best_state = {k: v for k, v in state.items() if k != "optimizer_state"}
                best_state["best_epoch"] = epoch
                torch.save(best_state, ckpts["best"])
                logger.info(f"New best checkpoint at epoch {epoch} ({select_by}={score:.4f})")

    plot_loss_curves(history, paths["figures"] / f"loss_curve_{model_name}.png", model_name)
    save_json(history, paths["outputs"] / f"history_{model_name}.json")
    best = torch.load(ckpts["best"], map_location="cpu", weights_only=True)
    summary = {
        "model": model_name, "type": "neural", "arch": model_cfg.get("arch", "meanpool_mlp"),
        "param_count": total_params, "trainable_params": trainable, "vocab_size": len(tokenizer),
        "best_epoch": best["best_epoch"], "select_by": select_by,
        "train_loss": history["epochs"][best["best_epoch"] - 1]["train_loss"],
        "val_loss": history["epochs"][best["best_epoch"] - 1]["val_loss"],
        "val_accuracy": history["epochs"][best["best_epoch"] - 1]["val_accuracy"],
        "final_train_loss": history["epochs"][-1]["train_loss"],
        "final_val_loss": history["epochs"][-1]["val_loss"],
        "grad_norm_max_overall": max(h["grad_norm_max"] for h in history["epochs"]),
        "nan_or_skipped_steps": nan_steps,
        "training_time_s": t_train.seconds,
        "train_examples_per_sec": seen / t_train.seconds,
        "epochs": model_cfg["epochs"],
        "checkpoint": rel(ckpts["best"]),
        "checkpoint_sha256": file_sha256(ckpts["best"]),
        "latest_checkpoint": rel(ckpts["latest"]),
    }
    return summary, device

In [ ]:
def train_model(cfg, model_name, smoke=False):
    """Train one configured model; returns the training summary dict."""
    if smoke:
        cfg = apply_smoke_overrides(cfg)
    paths = get_paths(cfg, smoke)
    model_cfg = get_model_cfg(cfg, model_name)
    log_path = paths["logs"] / f"train_{model_name}_{timestamp()}.log"
    logger = setup_logger("train", log_path)
    set_seed(cfg["seed"])
    hw = hardware_info()
    logger.info(f"Model '{model_name}' ({model_cfg['type']}) | smoke={smoke} | seed={cfg['seed']}")
    logger.info(f"Hardware: {hw}")
    logger.info(f"Model config: {model_cfg}")

    splits = prepare_data(cfg, paths, logger, smoke=smoke)
    if model_cfg["type"] == "logreg":
        summary, device = train_logreg(cfg, model_name, model_cfg, splits, paths, logger)
    elif model_cfg["type"] == "neural":
        summary, device = train_neural(cfg, model_name, model_cfg, splits, paths, logger)
    else:
        raise ValueError(f"Unknown model type {model_cfg['type']}")

    summary.update({
        "seed": cfg["seed"], "smoke": smoke, "hyperparameters": model_cfg,
        "train_device": device_label(hw, device),
        "hardware": hw, "train_peak_memory": peak_memory_mb(device),
        "train_log": rel(log_path), "n_train": len(splits["train"]), "n_val": len(splits["val"]),
    })
    save_json(summary, paths["outputs"] / f"train_summary_{model_name}.json")
    manifest = write_manifest(paths, model_name, "train", {"config": cfg, "summary": summary})
    logger.info(f"Summary: {to_builtin({k: v for k, v in summary.items() if k != 'hardware'})}")
    logger.info(f"Manifest written to {rel(manifest)}")
    return summary

In [ ]:
neural = [m for m in MODELS if RUN_CFG["models"][m]["type"] == "neural"]
keys = ["learning_rate", "weight_decay", "batch_size", "eval_batch_size", "epochs", "grad_clip", "dropout",
        "num_workers", "select_by"]
display(pd.DataFrame({m: {k: RUN_CFG["models"][m][k] for k in keys} for m in neural}))

## 12. Evaluation functions

- `compute_metrics`: accuracy; precision, recall, and F1 with macro, micro, and weighted averaging; ROC-AUC; PR-AUC; MCC; Brier score; expected calibration error; confusion matrix. The decision threshold is `threshold` on P(positive).
- `expected_calibration_error`: `ece_bins` equal-width bins over the confidence of the predicted class.
- `bootstrap_ci`: 95% percentile intervals over `bootstrap_samples` resamples of the test set, for accuracy, macro-F1, and MCC.
- `plot_confusion` / `plot_reliability`: confusion matrix and reliability diagram.
- `Predictor`: loads any of the three checkpoints with the preprocessing and vocabulary stored inside it and returns P(positive); used for evaluation and single-review prediction.

Cost metrics (parameters, training time, training and inference examples per second, peak GPU memory and peak CPU RSS) are recorded by `train_model` and `evaluate_model`.

In [ ]:
def expected_calibration_error(y_true, p_pos, n_bins=15):
    """Weighted gap between confidence and accuracy over equal-width confidence bins."""
    y_true = np.asarray(y_true)
    p_pos = np.asarray(p_pos, dtype=float)
    pred = (p_pos >= 0.5).astype(int)
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = (pred == y_true).astype(float)
    # Why: binary confidence lives in [0.5, 1], so the bins cover that range only.
    edges = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1], right=True), 0, n_bins - 1)
    ece, bins = 0.0, []
    for b in range(n_bins):
        m = idx == b
        if m.any():
            gap = abs(correct[m].mean() - conf[m].mean())
            ece += m.mean() * gap
            bins.append({"bin": b, "lo": edges[b], "hi": edges[b + 1], "count": int(m.sum()),
                         "accuracy": float(correct[m].mean()), "confidence": float(conf[m].mean())})
    return float(ece), bins


def _counts_metrics(tp, fp, fn, tn):
    """Accuracy, macro-F1, and MCC from confusion counts (arrays allowed)."""
    n = tp + fp + fn + tn
    acc = (tp + tn) / n
    f1_pos = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros_like(acc), where=(2 * tp + fp + fn) > 0)
    f1_neg = np.divide(2 * tn, 2 * tn + fn + fp, out=np.zeros_like(acc), where=(2 * tn + fn + fp) > 0)
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = np.divide(tp * tn - fp * fn, denom, out=np.zeros_like(acc), where=denom > 0)
    return acc, (f1_pos + f1_neg) / 2, mcc


def bootstrap_ci(y_true, y_pred, n_boot=1000, seed=0, alpha=0.05, chunk=100):
    """Percentile bootstrap CIs for accuracy, macro-F1, and MCC (resampling test reviews)."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    stats = {"accuracy": [], "macro_f1": [], "mcc": []}
    for start in range(0, n_boot, chunk):
        b = min(chunk, n_boot - start)
        idx = rng.integers(0, n, size=(b, n))
        t, p = y_true[idx], y_pred[idx]
        tp = ((t == 1) & (p == 1)).sum(1).astype(float)
        tn = ((t == 0) & (p == 0)).sum(1).astype(float)
        fp = ((t == 0) & (p == 1)).sum(1).astype(float)
        fn = ((t == 1) & (p == 0)).sum(1).astype(float)
        acc, mf1, mcc = _counts_metrics(tp, fp, fn, tn)
        stats["accuracy"].append(acc)
        stats["macro_f1"].append(mf1)
        stats["mcc"].append(mcc)
    out = {}
    for k, v in stats.items():
        v = np.concatenate(v)
        out[k] = (float(np.quantile(v, alpha / 2)), float(np.quantile(v, 1 - alpha / 2)))
    return out

In [ ]:
def compute_metrics(y_true, p_pos, threshold=0.5, ece_bins=15):
    y_true = np.asarray(y_true)
    p_pos = np.asarray(p_pos, dtype=float)
    y_pred = (p_pos >= threshold).astype(int)
    m = {"accuracy": accuracy_score(y_true, y_pred)}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=avg, labels=[0, 1],
                                                     zero_division=0)
        m.update({f"precision_{avg}": p, f"recall_{avg}": r, f"f1_{avg}": f})
    both = len(np.unique(y_true)) == 2
    m["roc_auc"] = roc_auc_score(y_true, p_pos) if both else float("nan")
    m["pr_auc"] = average_precision_score(y_true, p_pos) if both else float("nan")
    m["mcc"] = matthews_corrcoef(y_true, y_pred)
    m["brier"] = brier_score_loss(y_true, p_pos)
    m["ece"], bins = expected_calibration_error(y_true, p_pos, ece_bins)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    m.update({"tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1])})
    m = {k: float(v) if not isinstance(v, int) else v for k, v in m.items()}
    report = classification_report(y_true, y_pred, labels=[0, 1], zero_division=0,
                                   target_names=[LABEL_NAMES[0], LABEL_NAMES[1]], digits=4)
    return m, cm, report, bins, y_pred

In [ ]:
def plot_confusion(cm, path, title):
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["pred neg", "pred pos"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["true neg", "true pos"])
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def plot_reliability(bins, ece, path, title):
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.plot([0.5, 1], [0.5, 1], "k--", lw=1, label="perfect calibration")
    if bins:
        ax.plot([b["confidence"] for b in bins], [b["accuracy"] for b in bins], "o-", label="model")
    ax.set_xlabel("mean confidence")
    ax.set_ylabel("accuracy")
    ax.set_title(f"{title} (ECE={ece:.4f})")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)

In [ ]:
class Predictor:
    def __init__(self, cfg, model_name, paths, device=None):
        self.model_name = model_name
        self.model_cfg = get_model_cfg(cfg, model_name)
        self.type = self.model_cfg["type"]
        ck = checkpoint_paths(paths, model_name, self.type)
        if self.type == "logreg":
            if not ck["model"].exists():
                raise FileNotFoundError(f"{ck['model']} not found. Train '{model_name}' first.")
            self.device = torch.device("cpu")
            bundle = joblib.load(ck["model"])
            self.pipe = bundle["pipeline"]
            self.preprocessor = TextPreprocessor(bundle["preprocessing_cfg"])
        else:
            if not ck["best"].exists():
                raise FileNotFoundError(f"{ck['best']} not found. Train '{model_name}' first.")
            self.device = device or get_device()
            state = torch.load(ck["best"], map_location=self.device, weights_only=True)
            self.tokenizer = WordTokenizer(state["vocab"])
            self.max_length = state["tokenizer_cfg"]["max_length"]
            # Why: rebuild with the preprocessing and architecture saved inside the
            # checkpoint, so a later config edit cannot silently change a trained model.
            self.preprocessor = TextPreprocessor(state["preprocessing_cfg"])
            self.model = build_neural_model(state["model_cfg"], len(self.tokenizer)).to(self.device)
            self.model.load_state_dict(state["model_state"])
            self.model.eval()
            self.best_epoch = state.get("best_epoch")
            self.eval_batch_size = self.model_cfg.get("eval_batch_size", 512)
            self.seed = state["seed"]

    @torch.no_grad()
    def predict_proba_tokens(self, token_strings):
        """P(positive) for already-preprocessed token strings."""
        token_strings = list(token_strings)
        if self.type == "logreg":
            return self.pipe.predict_proba(token_strings)[:, 1]
        ds = EncodedReviews(token_strings, np.zeros(len(token_strings)), self.tokenizer, self.max_length)
        loader = make_loader(ds, self.eval_batch_size, False, self.seed)
        out = []
        for ids, mask, _ in loader:
            logits = self.model(ids.to(self.device), mask.to(self.device))
            out.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        return np.concatenate(out)

    def predict_texts(self, texts):
        tokens = [self.preprocessor.to_string(t) for t in texts]
        p_pos = self.predict_proba_tokens(tokens)
        results = []
        for p in p_pos:
            label = int(p >= 0.5)
            results.append({
                "label": label,
                "label_name": LABEL_NAMES[label],
                "confidence": float(max(p, 1 - p)),
                "probabilities": {"negative": float(1 - p), "positive": float(p)},
            })
        return results

## 13. Statistical comparison, slices, and error-review selection

- **McNemar's test** (`mcnemar_test`): each neural model against the baseline on the same test reviews. `b` counts reviews the baseline gets right and the other model gets wrong, `c` the reverse. With 25 or more discordant reviews the statistic is the continuity-corrected chi-square `(|b - c| - 1)^2 / (b + c)` with one degree of freedom; below 25 an exact binomial test is used.
- **Slices** (`slice_table`): accuracy, macro-F1, and error rate per slice of section 6.
- **Error review** (`select_error_review`): 20 misclassified test reviews chosen by a fixed rule: the `error_review_k` most confident false positives, the most confident false negatives, the errors closest to the threshold, and errors drawn with the run seed from the worst slice with at least `min_slice_n` reviews.

`evaluate_model` runs the whole evaluation of one model and writes the predictions, metrics, slices, McNemar result, disagreements, error-review candidates, figures, the `metrics_report.csv` row, and a manifest. McNemar needs the baseline's predictions, so the baseline is evaluated first.

In [ ]:
def mcnemar_test(y_true, pred_a, pred_b):
    """Paired test on the reviews where exactly one of the two models is correct."""
    ca = np.asarray(pred_a) == np.asarray(y_true)
    cb = np.asarray(pred_b) == np.asarray(y_true)
    b = int((ca & ~cb).sum())   # A right, B wrong
    c = int((~ca & cb).sum())   # A wrong, B right
    if b + c == 0:
        return {"b_a_right_b_wrong": b, "c_a_wrong_b_right": c, "statistic": 0.0,
                "p_value": 1.0, "method": "no discordant pairs"}
    if b + c < 25:
        # Why: with few discordant pairs the chi-square approximation is poor,
        # so fall back to the exact binomial version.
        p = binomtest(b, b + c, 0.5).pvalue
        return {"b_a_right_b_wrong": b, "c_a_wrong_b_right": c, "statistic": float(min(b, c)),
                "p_value": float(p), "method": "exact binomial"}
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    return {"b_a_right_b_wrong": b, "c_a_wrong_b_right": c, "statistic": float(stat),
            "p_value": float(chi2.sf(stat, 1)), "method": "chi2 with continuity correction"}


def slice_table(df, y_true, y_pred):
    rows = []
    for name, mask in slice_masks(df).items():
        n = int(mask.sum())
        if n == 0:
            rows.append({"slice": name, "n": 0, "macro_f1": np.nan, "error_rate": np.nan})
            continue
        rows.append({
            "slice": name, "n": n,
            "positive_fraction": float(y_true[mask].mean()),
            "macro_f1": float(f1_score(y_true[mask], y_pred[mask], average="macro", labels=[0, 1],
                                       zero_division=0)),
            "error_rate": float((y_true[mask] != y_pred[mask]).mean()),
        })
    return pd.DataFrame(rows)


def select_error_review(pred_df, slices, k, min_slice_n, seed):
    """Pick 5 confident FPs, 5 confident FNs, 5 near-threshold, 5 from the worst slice."""
    err = pred_df[~pred_df["correct"]]
    picked, groups = set(), []

    def take(frame, bucket):
        frame = frame[~frame["id"].isin(picked)].head(k).copy()
        frame.insert(0, "review_bucket", bucket)
        picked.update(frame["id"])
        groups.append(frame)

    take(err[err["error_type"] == "false_positive"].sort_values("confidence", ascending=False),
         "confident_false_positive")
    take(err[err["error_type"] == "false_negative"].sort_values("confidence", ascending=False),
         "confident_false_negative")
    take(err.assign(d=(err["prob_positive"] - 0.5).abs()).sort_values("d").drop(columns="d"),
         "near_threshold")
    cand = slices[(slices["slice"] != "all") & (slices["n"] >= min_slice_n)].dropna(subset=["error_rate"])
    worst = None
    if len(cand):
        worst = cand.sort_values("error_rate", ascending=False).iloc[0]["slice"]
        mask = pd.Series(slice_masks(pred_df)[worst], index=pred_df.index)
        pool = err[mask[err.index]]
        pool = pool[~pool["id"].isin(picked)]
        take(pool.sample(n=min(k, len(pool)), random_state=seed), f"slice:{worst}")
    out = pd.concat(groups, ignore_index=True) if groups else pd.DataFrame()
    out["manual_error_type"] = ""   # fill in by hand after reading each review
    out["proposed_testable_fix"] = ""
    return out, worst

In [ ]:
def evaluate_model(cfg, model_name, smoke=False):
    if smoke:
        cfg = apply_smoke_overrides(cfg)
    paths = get_paths(cfg, smoke)
    ev = cfg["evaluation"]
    model_cfg = get_model_cfg(cfg, model_name)
    log_path = paths["logs"] / f"eval_{model_name}_{timestamp()}.log"
    logger = setup_logger("eval", log_path)
    set_seed(cfg["seed"])
    hw = hardware_info()

    train_summary_path = paths["outputs"] / f"train_summary_{model_name}.json"
    if not train_summary_path.exists():
        raise FileNotFoundError(f"{rel(train_summary_path)} missing. Run train.py --model {model_name} first.")
    train_summary = load_json(train_summary_path)

    test = prepare_data(cfg, paths, logger, smoke=smoke)["test"]
    predictor = Predictor(cfg, model_name, paths)
    device = predictor.device
    y = test["label"].to_numpy()

    reset_peak_memory(device)
    sync(device)
    with Timer() as t:
        p_pos = predictor.predict_proba_tokens(test["tokens"])
        sync(device)
    infer_mem = peak_memory_mb(device)
    logger.info(f"Inference on {len(test)} reviews took {t.seconds:.2f}s on {device_label(hw, device)}")

    metrics, cm, report, bins, y_pred = compute_metrics(y, p_pos, ev["threshold"], ev["ece_bins"])
    ci = bootstrap_ci(y, y_pred, ev["bootstrap_samples"], cfg["seed"])
    logger.info("Metrics: " + ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                                        for k, v in metrics.items()))
    logger.info(f"Bootstrap 95% CIs ({ev['bootstrap_samples']} resamples): {ci}")
    logger.info(f"Classification report:\n{report}")

    # ---- per-review predictions
    conf = np.maximum(p_pos, 1 - p_pos)
    pred_df = test[["id", "text", "label", "n_words_raw", "n_tokens", "length_bucket",
                    "has_negation", "has_contrast"]].rename(columns={"label": "true_label"}).copy()
    pred_df["predicted_label"] = y_pred
    pred_df["prob_positive"] = p_pos
    pred_df["confidence"] = conf
    pred_df["correct"] = pred_df["true_label"] == pred_df["predicted_label"]
    pred_df["error_type"] = np.where(pred_df["correct"], "correct",
                                     np.where(pred_df["predicted_label"] == 1, "false_positive",
                                              "false_negative"))
    pred_df["model_name"] = model_name
    out = paths["outputs"]
    pred_df.to_csv(out / f"predictions_{model_name}.csv", index=False)

    mis = pred_df[~pred_df["correct"]].sort_values("confidence", ascending=False)
    mis_cols = ["id", "text", "true_label", "predicted_label", "confidence", "prob_positive",
                "model_name", "error_type", "length_bucket", "has_negation", "has_contrast", "n_words_raw"]
    mis[mis_cols].to_csv(out / f"misclassified_examples_{model_name}.csv", index=False)
    combined = pd.concat([pd.read_csv(f) for f in sorted(out.glob("misclassified_examples_*.csv"))],
                         ignore_index=True)
    combined.to_csv(out / "misclassified_examples.csv", index=False)

    # ---- slices and error review sample
    slices = slice_table(pred_df, y, y_pred)
    slices.insert(0, "model", model_name)
    slices.to_csv(out / f"slice_metrics_{model_name}.csv", index=False)
    review, worst = select_error_review(pred_df, slices, ev["error_review_k"],
                                        ev["min_slice_n"], cfg["seed"])
    review.to_csv(out / f"error_review_candidates_{model_name}.csv", index=False)
    logger.info(f"Error review sample: {len(review)} rows (worst slice: {worst})")

    # ---- McNemar against the baseline, using identical test ids
    mcn = {}
    base = ev["baseline_model"]
    base_pred_path = out / f"predictions_{base}.csv"
    if model_name != base and base_pred_path.exists():
        bdf = pd.read_csv(base_pred_path, usecols=["id", "predicted_label"])
        merged = pred_df[["id", "text", "true_label", "predicted_label", "confidence"]].merge(
            bdf.rename(columns={"predicted_label": "baseline_pred"}), on="id", how="inner")
        if len(merged) != len(pred_df):
            logger.warning("Baseline predictions cover a different test set; McNemar uses the overlap only")
        mcn = mcnemar_test(merged["true_label"], merged["baseline_pred"], merged["predicted_label"])
        mcn.update({"baseline": base, "model": model_name, "n": len(merged)})
        upsert_csv_row(out / "mcnemar_results.csv", mcn, ["baseline", "model"])
        base_right = merged["baseline_pred"] == merged["true_label"]
        model_right = merged["predicted_label"] == merged["true_label"]
        merged["disagreement"] = np.where(base_right & ~model_right, f"{base}_right_{model_name}_wrong",
                                          np.where(~base_right & model_right,
                                                   f"{base}_wrong_{model_name}_right", "same"))
        merged[merged["disagreement"] != "same"].to_csv(
            out / f"disagreements_{base}_vs_{model_name}.csv", index=False)
        logger.info(f"McNemar vs {base}: {mcn}")
    elif model_name != base:
        logger.warning(f"No {rel(base_pred_path)} yet; evaluate the baseline first to get McNemar")

    # ---- figures
    plot_confusion(cm, paths["figures"] / f"confusion_matrix_{model_name}.png", model_name)
    plot_reliability(bins, metrics["ece"], paths["figures"] / f"reliability_{model_name}.png", model_name)

    # ---- one row in metrics_report.csv per model
    row = {
        "model": model_name, "type": model_cfg["type"],
        "arch": model_cfg.get("arch", "tfidf_logreg" if model_cfg["type"] == "logreg" else "meanpool_mlp"),
        "hyperparameters": str(train_summary["hyperparameters"]),
        "seed": cfg["seed"], "n_test": len(test),
        **metrics,
        "accuracy_ci95_low": ci["accuracy"][0], "accuracy_ci95_high": ci["accuracy"][1],
        "macro_f1_ci95_low": ci["macro_f1"][0], "macro_f1_ci95_high": ci["macro_f1"][1],
        "mcc_ci95_low": ci["mcc"][0], "mcc_ci95_high": ci["mcc"][1],
        "mcnemar_vs_baseline_p": mcn.get("p_value", np.nan),
        "mcnemar_vs_baseline_stat": mcn.get("statistic", np.nan),
        "param_count": train_summary["param_count"],
        "train_loss": train_summary["train_loss"], "val_loss": train_summary["val_loss"],
        "training_time_s": train_summary["training_time_s"],
        "train_examples_per_sec": train_summary["train_examples_per_sec"],
        "inference_time_s": t.seconds,
        "inference_examples_per_sec": len(test) / t.seconds,
        "train_peak_gpu_mb": train_summary["train_peak_memory"]["gpu_peak_mb"],
        "train_peak_cpu_rss_mb": train_summary["train_peak_memory"]["cpu_peak_rss_mb"],
        "eval_peak_gpu_mb": infer_mem["gpu_peak_mb"],
        "eval_peak_cpu_rss_mb": infer_mem["cpu_peak_rss_mb"],
        "train_device": train_summary["train_device"],
        "eval_device": device_label(hw, device),
        "checkpoint": train_summary["checkpoint"],
        "checkpoint_sha256": train_summary["checkpoint_sha256"],
        "train_log": train_summary["train_log"], "eval_log": rel(log_path),
    }
    for _, s in slices.iterrows():
        row[f"slice_{s['slice']}_n"] = s["n"]
        row[f"slice_{s['slice']}_macro_f1"] = s["macro_f1"]
        row[f"slice_{s['slice']}_error_rate"] = s["error_rate"]
    upsert_csv_row(paths["metrics_report"], row, ["model"])

    full = {"row": row, "bootstrap_ci95": ci, "mcnemar": mcn, "calibration_bins": bins,
            "confusion_matrix": cm, "classification_report": report, "worst_slice": worst}
    save_json(full, out / f"metrics_{model_name}.json")
    write_manifest(paths, model_name, "eval", {"config": cfg, "metrics": row})
    logger.info(f"Wrote {rel(paths['metrics_report'])} and outputs for {model_name}")
    return row

## 14. Execute the experiments

Train the three models in config order, then evaluate them in the same order (baseline first). Each call receives a fresh copy of `CONFIG`, as each command-line run loaded a fresh config. With `RUN_MODE = "full"` this is the reported experiment.

Peak CPU RSS is measured over the whole process. The reported values come from separate command-line runs, one per model; in a notebook kernel the value also includes everything loaded earlier, so it is not comparable with them.

In [ ]:
train_summaries = {}
for m in MODELS:
    train_summaries[m] = train_model(copy.deepcopy(CONFIG), m, smoke=SMOKE)

In [ ]:
eval_rows = {}
for m in MODELS:
    eval_rows[m] = evaluate_model(copy.deepcopy(CONFIG), m, smoke=SMOKE)

## 15. Results and model comparison

Everything below is read from the files this notebook's run wrote.

In [ ]:
OUT = PATHS["outputs"]
FIG = PATHS["figures"]
report = pd.read_csv(PATHS["metrics_report"]).set_index("model").loc[MODELS]
metrics_json = {m: load_json(OUT / f"metrics_{m}.json") for m in MODELS}

SUMMARY_COLS = ["type", "param_count", "vocab_size", "best_epoch", "train_loss", "val_loss", "val_accuracy",
                "training_time_s", "train_examples_per_sec", "grad_norm_max_overall", "nan_or_skipped_steps",
                "n_train", "n_val", "train_device"]
rows = {}
for m, s in train_summaries.items():
    row = {c: s.get(c) for c in SUMMARY_COLS}
    row["train peak GPU MB"] = s["train_peak_memory"]["gpu_peak_mb"]
    row["train peak CPU RSS MB"] = s["train_peak_memory"]["cpu_peak_rss_mb"]
    rows[m] = row
display(pd.DataFrame(rows).astype(str))

In [ ]:
for m in neural:
    h = load_json(OUT / f"history_{m}.json")
    display(Markdown(f"**{m}**: per-epoch history"))
    display(pd.DataFrame(h["epochs"]).round(4))
    display(Image(filename=str(FIG / f"loss_curve_{m}.png")))

In [ ]:
METRIC_COLS = ["n_test", "accuracy", "precision_macro", "recall_macro", "f1_macro", "precision_micro", "recall_micro",
               "f1_micro", "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc", "pr_auc", "mcc",
               "brier", "ece", "accuracy_ci95_low", "accuracy_ci95_high", "macro_f1_ci95_low", "macro_f1_ci95_high",
               "mcc_ci95_low", "mcc_ci95_high"]
COST_COLS = ["param_count", "training_time_s", "train_examples_per_sec", "inference_time_s",
             "inference_examples_per_sec", "train_peak_gpu_mb", "train_peak_cpu_rss_mb", "eval_peak_gpu_mb",
             "eval_peak_cpu_rss_mb", "train_device", "eval_device"]
display(report[METRIC_COLS].T)
display(report[COST_COLS].T)

In [ ]:
display(report[["tn", "fp", "fn", "tp"]])
for m in MODELS:
    display(Markdown(f"**{m}**"))
    print(metrics_json[m]["classification_report"])
    display(Image(filename=str(FIG / f"confusion_matrix_{m}.png")))
    display(Image(filename=str(FIG / f"reliability_{m}.png")))

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 4.5))
for m in MODELS:
    p = pd.read_csv(OUT / f"predictions_{m}.csv", usecols=["true_label", "prob_positive"])
    fpr, tpr, _ = roc_curve(p["true_label"], p["prob_positive"])
    prec, recall, _ = precision_recall_curve(p["true_label"], p["prob_positive"])
    ax_roc.plot(fpr, tpr, label=f"{m} (AUC {report.loc[m, 'roc_auc']:.4f})")
    ax_pr.plot(recall, prec, label=f"{m} (AP {report.loc[m, 'pr_auc']:.4f})")
ax_roc.plot([0, 1], [0, 1], "k--", lw=0.8)
ax_roc.set(xlabel="false positive rate", ylabel="true positive rate", title="ROC, test set")
ax_pr.set(xlabel="recall (positive class)", ylabel="precision", title="Precision-recall, test set")
for ax in (ax_roc, ax_pr):
    ax.legend()
    ax.grid(alpha=0.3)
show_fig(fig)

### McNemar's test and slices

In [ ]:
display(pd.read_csv(OUT / "mcnemar_results.csv"))
for m in MODELS:
    if m == BASELINE:
        continue
    d = pd.read_csv(OUT / f"disagreements_{BASELINE}_vs_{m}.csv")
    print(f"{BASELINE} vs {m}: {len(d):,} test reviews with different predictions;",
          d["disagreement"].value_counts().to_dict())

In [ ]:
slices = pd.concat([pd.read_csv(OUT / f"slice_metrics_{m}.csv") for m in MODELS])
order = list(dict.fromkeys(slices["slice"]))
display(slices[slices["model"] == MODELS[0]].set_index("slice")[["n", "positive_fraction"]].loc[order])
display(slices.pivot(index="slice", columns="model", values="macro_f1").loc[order, MODELS].round(4)
        .rename(columns=lambda m: f"macro-F1 {m}"))
error_rates = slices.pivot(index="slice", columns="model", values="error_rate").loc[order, MODELS]
display(error_rates.round(4).rename(columns=lambda m: f"error rate {m}"))

fig, ax = plt.subplots(figsize=(10, 4))
error_rates.plot.bar(ax=ax)
ax.set_ylabel("error rate")
ax.set_title("Error rate by slice (test set)")
ax.grid(axis="y", alpha=0.3)
show_fig(fig)

### Model comparison

The main metrics side by side, with the best model per metric (higher is better except for Brier score and ECE).

In [ ]:
HIGHER = ["accuracy", "f1_macro", "roc_auc", "pr_auc", "mcc"]
LOWER = ["brier", "ece"]
comparison = report[HIGHER + LOWER].T.astype(float)
comparison["best"] = [comparison.loc[k, MODELS].idxmax() if k in HIGHER else comparison.loc[k, MODELS].idxmin()
                      for k in comparison.index]
ci = pd.DataFrame({m: {
    "accuracy 95% CI": f"[{report.loc[m, 'accuracy_ci95_low']:.4f}, {report.loc[m, 'accuracy_ci95_high']:.4f}]",
    "macro-F1 95% CI": f"[{report.loc[m, 'macro_f1_ci95_low']:.4f}, {report.loc[m, 'macro_f1_ci95_high']:.4f}]",
    "MCC 95% CI": f"[{report.loc[m, 'mcc_ci95_low']:.4f}, {report.loc[m, 'mcc_ci95_high']:.4f}]",
    "McNemar p vs baseline": report.loc[m, "mcnemar_vs_baseline_p"],
} for m in MODELS})
display(comparison.round(4))
display(ci)

### Single-review predictions

In [ ]:
demo_reviews = [
    "The pasta was delicious and the staff were friendly. We will be back!",
    "Waited an hour for cold food and the manager was rude. Never again.",
    "Not bad at all, but the prices are a bit high for what you get.",
]
rows = []
for m in MODELS:
    for text, r in zip(demo_reviews, Predictor(copy.deepcopy(RUN_CFG), m, PATHS).predict_texts(demo_reviews)):
        rows.append((m, text, r["label_name"], round(r["probabilities"]["positive"], 4)))
show_table(rows, columns=["model", "review", "predicted", "P(positive)"])

### Comparison with the reported runs

When the reported `metrics_report.csv` is present, the absolute difference of every metric from the reported value. The neural models use deterministic algorithms, but results can still differ slightly across library versions and GPUs.

In [ ]:
REPORTED_REPORT = MEMBER_DIR / "metrics_report.csv"


def compare_to_reported(new_report_path, label):
    reported = pd.read_csv(REPORTED_REPORT).set_index("model").loc[MODELS]
    new = pd.read_csv(new_report_path).set_index("model").loc[MODELS]
    cols = [c for c in METRIC_COLS + ["tn", "fp", "fn", "tp", "mcnemar_vs_baseline_stat", "mcnemar_vs_baseline_p"]
            if c in new.columns and c in reported.columns]
    diff = (new[cols].astype(float) - reported[cols].astype(float)).abs()
    print(f"largest absolute difference from the reported metrics ({label}): {np.nanmax(diff.to_numpy()):.3g}")
    return diff.T.rename(columns=lambda m: f"|{label} - reported| {m}")


if REPORTED_REPORT.exists() and not SMOKE:
    display(compare_to_reported(PATHS["metrics_report"], "this notebook"))
else:
    print("Skipped (smoke mode or no reported metrics_report.csv).")

### Re-evaluating the reported checkpoints (`VERIFY_REPORTED_CHECKPOINTS`)

Copies the reported checkpoints and training summaries from `task2_sentiment/parth/` into `notebook_runs/<RUN_TAG>/verify_reported/`, reuses this run's processed-data cache (same cache key), runs `evaluate_model` for all three models, and compares every metric with the reported `metrics_report.csv`.

In [ ]:
REPORTED_CKPTS = {m: MEMBER_DIR / "checkpoints" / Path(report.loc[m, "checkpoint"]).name for m in MODELS}
REPORTED_SUMMARIES = {m: MEMBER_DIR / "outputs" / f"train_summary_{m}.json" for m in MODELS}
have_reported = all(p.exists() for p in [*REPORTED_CKPTS.values(), *REPORTED_SUMMARIES.values(), REPORTED_REPORT])
verify_rows = None
if VERIFY_REPORTED_CHECKPOINTS and not SMOKE and have_reported:
    verify_cfg = copy.deepcopy(CONFIG)
    verify_cfg["paths"]["member_dir"] = rel(RUN_DIR / "verify_reported")
    verify_cfg["paths"]["smoke_dir"] = rel(RUN_DIR / "verify_reported" / "smoke_runs")
    verify_paths = get_paths(verify_cfg)
    for m in MODELS:
        shutil.copy2(REPORTED_CKPTS[m], verify_paths["checkpoints"])
        shutil.copy2(REPORTED_SUMMARIES[m], verify_paths["outputs"])
    shutil.copytree(PATHS["data_processed"], verify_paths["data_processed"], dirs_exist_ok=True)
    verify_rows = {m: evaluate_model(copy.deepcopy(verify_cfg), m) for m in MODELS}
    display(compare_to_reported(verify_paths["metrics_report"], "re-evaluated"))
else:
    print("Skipped (flag off, smoke mode, or the reported checkpoints / summaries / metrics_report.csv are not present).")

## 16. Error analysis: 20 misclassified test reviews

The reviewed model is `neural_cnn`, as in `failure_analysis.md`. `select_error_review` (section 13) picked the 20 errors with a fixed rule: the 5 most confident false positives, the 5 most confident false negatives, the 5 errors closest to the 0.5 threshold, and 5 errors drawn with the run seed from the worst slice with at least `min_slice_n` reviews.

In [ ]:
REVIEW_MODEL = "neural_cnn"
cands = pd.read_csv(OUT / f"error_review_candidates_{REVIEW_MODEL}.csv")
assert not cands["correct"].any()
if not SMOKE:
    assert len(cands) == 4 * CONFIG["evaluation"]["error_review_k"]
print(cands["review_bucket"].value_counts(sort=False).to_dict())
cands.insert(0, "case", range(1, len(cands) + 1))
with pd.option_context("display.precision", 10):
    display(cands[["case", "review_bucket", "id", "true_label", "predicted_label", "prob_positive", "confidence",
                   "error_type", "n_words_raw", "n_tokens", "length_bucket", "has_negation", "has_contrast"]])

reported_cands_path = MEMBER_DIR / "outputs" / f"error_review_candidates_{REVIEW_MODEL}.csv"
if reported_cands_path.exists() and not SMOKE:
    reported_ids = pd.read_csv(reported_cands_path)["id"].tolist()
    print("Same 20 reviews, in the same order, as the reported selection (used in the table below):",
          reported_ids == cands["id"].tolist())

In [ ]:
for _, r in cands.iterrows():
    print(f"Case {r['case']} | {r['review_bucket']} | {r['id']} | true {r['true_label']} "
          f"({LABEL_NAMES[r['true_label']]}) | predicted {r['predicted_label']} | "
          f"P(positive) {r['prob_positive']} | confidence {r['confidence']}")
    print(r["text"], "\n")

### Manual review

The review IDs are those of the reported selection; the cell above checks that this run picked the same reviews, and the cell before it prints every review in full.

Each row gives a short error-type label and one testable fix, framed as an experiment. The full reasoning for each case, with quotes from the review, is in `failure_analysis.md`.

| Case | Bucket | Review ID | Manual error type | Proposed testable fix |
|---|---|---|---|---|
| 1 | confident_false_positive | `test_29330` | Possible rating-text mismatch | Label audit of all-model confident errors; accuracy with and without flagged reviews |
| 2 | confident_false_positive | `test_5752` | Temporal sentiment shift | Update-marker feature or separate first-sentence score; error on reviews with NOTE/EDIT/UPDATE |
| 3 | confident_false_positive | `test_18122` | Contextual sentiment, ambiguous label | Closure-review subset: audit labels, measure per-model error |
| 4 | confident_false_positive | `test_37771` | Mixed sentiment / contrast | Keep contrast words in preprocessing; has_contrast error |
| 5 | confident_false_positive | `test_12480` | Possible rating-text mismatch | Same label audit as case 1 |
| 6 | confident_false_negative | `test_30793` | Temporal sentiment shift | Order-aware model; error on reviews with temporal markers |
| 7 | confident_false_negative | `test_22807` | Possible rating-text mismatch | Audit labels of EDIT/UPDATE reviews; accuracy with and without them |
| 8 | confident_false_negative | `test_33573` | Irony / unusual phrasing | Hand-label all-model errors as irony / label noise / genuine error |
| 9 | confident_false_negative | `test_9438` | Sentiment target shift | Annotated other-business subset; compare with an order-aware model |
| 10 | confident_false_negative | `test_11398` | Mixed sentiment / contrast | Sentence-level pooling; error on short has_contrast reviews |
| 11 | near_threshold | `test_34487` | Negation-heavy, TextCNN-only | Negation-scope marking; has_negation error |
| 12 | near_threshold | `test_32335` | Mixed sentiment / contrast | Average logreg and TextCNN probabilities; accuracy for P in [0.4, 0.6] |
| 13 | near_threshold | `test_1372` | Implicit sentiment | Add the TF-IDF score as a TextCNN input; error on reviews without lexicon words |
| 14 | near_threshold | `test_32304` | Mixed sentiment / contrast | Validation-tuned threshold; accuracy for P in [0.4, 0.6] |
| 15 | near_threshold | `test_23084` | Mixed sentiment / contrast | Keep contrast words; error on near-threshold has_contrast reviews |
| 16 | slice:has_contrast | `test_10621` | Mixed sentiment / contrast | Contrast-clause features (split at contrast words); has_contrast error |
| 17 | slice:has_contrast | `test_8361` | Contextual sentiment, ambiguous label | Closure subset labelled two ways; per-model accuracy against each |
| 18 | slice:has_contrast | `test_20227` | Weak or balanced long review | Error by original star rating |
| 19 | slice:has_contrast | `test_8938` | Weak or balanced long review | Rating token for "N stars"; error on reviews stating a star count |
| 20 | slice:has_contrast | `test_34683` | Mixed sentiment / contrast | Separate final-sentence representation; error on verdict-at-end reviews |

### Patterns across the 20 cases

- **Confident false positives (cases 1 to 5):** all three models are confidently wrong. Cases 1 and 5 read as positive throughout (possible rating-text mismatch); cases 2, 3 and 4 hold long stretches of genuine praise with the negative verdict in one or two sentences.
- **Confident false negatives (cases 6 to 10):** positive reviews full of negative words that are not the verdict (the old owners, other malls, irony, a side complaint); case 7 reads as clearly negative, so its label is a likely mismatch.
- **Near-threshold errors (cases 11 to 15):** mixed or understated reviews where the TextCNN sits within 0.001 of 0.5; in all five the logistic regression is right, often confidently.
- **Worst-slice errors (cases 16 to 20, `has_contrast`):** reviews with both polarities where the verdict sits in one clause; the contrast words that mark it are removed by the stopword list.
- **Most common error type, and the fix to test first:** mixed sentiment / contrast (7 of 20, partly because 5 cases come from the contrast slice). The first experiment would retrain with contrast words and intensifiers kept in preprocessing and compare `has_contrast` error rates; a label audit of the all-model errors is the second, since 5 cases point to labels the text does not support.

## 17. Strengths, weaknesses, and limitations

The first table collects the facts for the comparison: overall quality, calibration, each model's worst slice (slices with at least `min_slice_n` reviews), cost and the McNemar p-value against the baseline. The second counts how often the three models are right or wrong together.

In [ ]:
evidence = {}
for m in MODELS:
    sl = slices[(slices["model"] == m) & (slices["slice"] != "all") & (slices["n"] >= RUN_CFG["evaluation"]["min_slice_n"])]
    worst = sl.sort_values("error_rate", ascending=False).iloc[0]
    evidence[m] = {
        "accuracy": report.loc[m, "accuracy"],
        "macro-F1": report.loc[m, "f1_macro"],
        "ECE": report.loc[m, "ece"],
        "Brier": report.loc[m, "brier"],
        "worst slice": worst["slice"],
        "worst-slice error rate": worst["error_rate"],
        "parameters": report.loc[m, "param_count"],
        "training time (s)": report.loc[m, "training_time_s"],
        "inference examples/s": report.loc[m, "inference_examples_per_sec"],
        "McNemar p vs baseline": report.loc[m, "mcnemar_vs_baseline_p"],
    }
display(pd.DataFrame(evidence).astype(str))

joined = None
for m in MODELS:
    d = pd.read_csv(OUT / f"predictions_{m}.csv", usecols=["id", "true_label", "predicted_label"]).rename(columns={"predicted_label": m})
    joined = d if joined is None else joined.merge(d.drop(columns="true_label"), on="id")
right = pd.DataFrame({m: joined[m] == joined["true_label"] for m in MODELS})
pattern = right.apply(lambda r: ", ".join(m for m in MODELS if r[m]) or "none", axis=1)
display(pattern.value_counts().rename_axis("models that are correct").rename("test reviews").to_frame())

- **Strengths and weaknesses of each model:** the logistic regression is the most accurate and cheapest model and reads the whole review, but its probabilities are underconfident (ECE 0.0321) and it sees word order only through bigrams. The MeanPoolMLP is the best calibrated (ECE 0.0048) and fastest at inference, but the least accurate and worst on every slice, which fits a representation that ignores order. The TextCNN is the best neural model (accuracy 0.9379) with good calibration, but it is the slowest to train and has the largest error gap on truncated reviews (8.89% against 6.17%).
- **Why the best model wins, and whether the gaps are significant and practically meaningful:** sparse unigram and bigram TF-IDF features already capture most polarity cues in 532K training reviews, while the neural models learn 6.4M embedding parameters from scratch in 5 epochs and lose some text to truncation. The logistic regression's 95% accuracy interval [0.9441, 0.9484] does not overlap either neural interval, and McNemar gives p = 2.39e-55 (b = 947, c = 376) and p = 2.81e-18 (b = 834, c = 513), strong evidence that the error patterns differ. In practical terms the gap to the TextCNN is 0.84 accuracy points, or 321 more errors out of 38,000 reviews.
- **Calibration:** ECE compares confidence with observed accuracy and says nothing about which model is right more often. The logistic regression is underconfident in all 15 bins (signed gap -0.0321); one plausible reason, not tested, is that L2 regularization pulls its probabilities toward 0.5. The neural models were trained with cross-entropy and selected by validation loss, and their confidence tracks accuracy closely (+0.0012 and +0.0032), so their ECE is lower even though their accuracy is too.
- **Accuracy versus cost:** the logistic regression has 200,001 parameters and trains in 58 s on the CPU, against 6.4M to 6.6M parameters and 73 s (MeanPoolMLP) or 127 s (TextCNN) on an RTX 5090, so the most accurate model is also the cheapest to train. All three score the 38,000 test reviews in under 2 s.
- **Limitations of the data, preprocessing, models and evaluation:** one seed and one split; neural inputs are truncated at 256 tokens; embeddings are learned from scratch with no pretrained model; the stock stopword list also removes contrast words, intensifiers and number words; mean pooling discards order and the TextCNN sees at most 5 tokens per filter; 5 epochs at a constant learning rate while validation loss was still falling; slices are keyword proxies, and everything is measured on the single official test set.

## 18. Future work

Each item is an experiment tied to an observed error or slice result; none has been run.

1. **Keep contrast words and intensifiers.** Retrain all three models with "but", "however", "though", "yet", "except", "only" and "very" removed from the stopword list, and compare error rates on the `has_contrast` slice (cases 4, 10, 16 and 20).
2. **Longer neural inputs.** Retrain the TextCNN with `max_length` 512 and compare its error rate on the 540 test reviews that are truncated at 256 (8.89% now).
3. **Order-aware or sentence-level model.** Add a model that keeps order across the review, for example a BiLSTM or attention pooling over sentence vectors, and compare it with the MeanPoolMLP and TextCNN on `has_contrast` and on reviews with temporal markers such as "now", "used to", "EDIT" or "update" (cases 2 and 6).
4. **Label audit.** Manually audit a random sample of the 1,355 reviews all three models get wrong, flag suspected rating-text mismatches (cases 1, 5 and 7), and report accuracy with and without them.
5. **Probability averaging.** Average the logistic regression and TextCNN probabilities with weights chosen on the validation set, and measure accuracy, ECE and the error rate among reviews with P(positive) between 0.4 and 0.6 (cases 11 to 15).

## 19. Summary

In [ ]:
final = pd.DataFrame({m: {
    "accuracy": f"{report.loc[m, 'accuracy']:.4f}",
    "macro-F1 (95% CI)": f"{report.loc[m, 'f1_macro']:.4f} [{report.loc[m, 'macro_f1_ci95_low']:.4f}, "
                         f"{report.loc[m, 'macro_f1_ci95_high']:.4f}]",
    "MCC": f"{report.loc[m, 'mcc']:.4f}",
    "ROC-AUC": f"{report.loc[m, 'roc_auc']:.4f}",
    "ECE": f"{report.loc[m, 'ece']:.4f}",
    "parameters": f"{int(report.loc[m, 'param_count']):,}",
    "training time (s)": f"{report.loc[m, 'training_time_s']:.1f}",
    "device": report.loc[m, "train_device"],
} for m in MODELS})
display(final)
print(f"test reviews: {int(report['n_test'].iloc[0]):,} | train / val reviews: "
      f"{train_summaries[MODELS[0]]['n_train']:,} / {train_summaries[MODELS[0]]['n_val']:,} | outputs: {rel(PATHS['root'])}")

- Sections 3 to 13 contain the complete implementation: loading, cleaning, preprocessing, vocabulary, slices and EDA, the three models, training, evaluation, and the statistical comparison.
- Section 14 runs the experiments; sections 15 to 17 and 19 show the results from the files in the run folder, where the raw logs and manifests are as well.